# Setup 02 — Generate dimensions
Part of the retail-demo setup utility. Re-runnable (overwrite-by-design).

Requires setup-01 (dictionaries seeded under `Files/setup/dictionaries/`).
The engine cell below is generated by `scripts/build_notebooks.py` — edit
the package sources under `utility/src/retail_setup/`, never the cell.

In [ ]:
# Fabric's Spark runtime ships an OLD typing_extensions and no pydantic, while the
# inlined engine below imports pydantic. Install both on the driver, then make the
# new typing_extensions actually load. Why each step:
#  - subprocess pip (not %pip): a %pip kernel restart tears down the Spark session
#    mid pipeline run and fails the activity.
#  - pin pydantic <2.12: 2.12+ imports typing_extensions.Sentinel.
#  - drop the stale modules from sys.modules: the runtime already imported the old
#    typing_extensions, so the freshly installed copy is shadowed; evicting it lets
#    the engine cell re-import the upgraded versions (pydantic needs symbols such
#    as TypeIs that the runtime's typing_extensions lacks).
#  - pydantic runs only on the driver (engine builds rows, then
#    spark.createDataFrame), so a driver-side install is sufficient.
import importlib
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "typing_extensions>=4.12.2", "pydantic>=2,<2.12",
])
for _name in [m for m in list(sys.modules)
              if m.split(".", 1)[0] in {"typing_extensions", "typing_inspection",
                                        "pydantic", "pydantic_core"}]:
    del sys.modules[_name]
importlib.invalidate_caches()

In [ ]:
# PARAMETERS — rendered by `retail-setup render`; defaults work unrendered
def _param(value: str, default: str) -> str:
    return default if len(value) > 1 and value[0] == value[1] == "{" else value

LAKEHOUSE_NAME = _param("{{LAKEHOUSE_NAME}}", "retail_lakehouse")
SILVER_DB = _param("{{SILVER_DB}}", "silver")
GOLD_DB = _param("{{GOLD_DB}}", "gold")
STORE_TYPE = _param("{{STORE_TYPE}}", "supercenter")
START_DATE = _param("{{START_DATE}}", "2025-01-01")
END_DATE = _param("{{END_DATE}}", "2025-03-31")
STORE_COUNT = int(_param("{{STORE_COUNT}}", "50"))
SEED = int(_param("{{SEED}}", "42"))
DICTIONARY_REF = _param("{{DICTIONARY_REF}}", "main")

spark.conf.set("spark.sql.session.timeZone", "UTC")  # engine timestamps depend on it
# Reorder seeded-draw math ((xxhash64(...)/1e12)*k -> (xxhash64(...)*k)/1e12) to
# cut rounding-error propagation and silence Fabric's Spark advisor. The rewrite
# shifts draws by <=1 ULP (generation is statistical, no exact-value contracts);
# set in every engine driver so dims and facts stay mutually consistent.
spark.conf.set("spark.advise.divisionExprConvertRule.enable", "true")

In [ ]:
# ENGINE SOURCE (generated — do not edit)
# Built by scripts/build_notebooks.py from utility/src/retail_setup/.

# --- retail_setup/dictionaries/models.py ---
"""Pydantic models for dictionary JSON files.

Entry models mirror the field names in
datagen/src/retail_datagen/shared/models.py (PascalCase) so the supercenter
conversion is verifiable 1:1. StoreTypeProfile is new: the behavioral knobs
that differentiate store types.
"""

import re
from decimal import Decimal

from pydantic import BaseModel, Field, field_validator


class TaxJurisdictionEntry(BaseModel):
    StateCode: str = Field(..., min_length=2, max_length=2)
    County: str = Field(..., min_length=1)
    City: str = Field(..., min_length=1)
    CombinedRate: Decimal = Field(..., ge=0, le=Decimal("0.20"))

    @field_validator("StateCode")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("StateCode must be alphabetic")
        return v.upper()

    @field_validator("CombinedRate", mode="before")
    @classmethod
    def _rate_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("CombinedRate must be a valid decimal number") from None


class GeographyEntry(BaseModel):
    City: str = Field(..., min_length=1)
    State: str = Field(..., min_length=2, max_length=2)
    Zip: str
    District: str = Field(..., min_length=1)
    Region: str = Field(..., min_length=1)

    @field_validator("State")
    @classmethod
    def _state_upper(cls, v: str) -> str:
        if not v.isalpha():
            raise ValueError("State must be alphabetic")
        return v.upper()

    @field_validator("Zip")
    @classmethod
    def _zip_format(cls, v: str) -> str:
        if not re.match(r"^\d{5}(-\d{4})?$", v):
            raise ValueError("Zip must be 12345 or 12345-6789")
        return v


class NameEntry(BaseModel):
    """One first or last name (shared across store types).

    The ``Name`` field intentionally unifies the separate ``FirstNameDict`` and
    ``LastNameDict`` models from datagen — both source models carry a single
    name string, so one model suffices here.
    """

    Name: str = Field(..., min_length=1)


class ProductBrandEntry(BaseModel):
    Brand: str = Field(..., min_length=1)
    Company: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)


class ProductEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    BasePrice: Decimal = Field(..., gt=0)
    Department: str = Field(..., min_length=1)
    Category: str = Field(..., min_length=1)
    Subcategory: str = Field(..., min_length=1)
    Tags: str | None = None

    @field_validator("BasePrice", mode="before")
    @classmethod
    def _price_decimal(cls, v) -> Decimal:
        try:
            return Decimal(str(v))
        except Exception:
            raise ValueError("BasePrice must be a valid decimal number") from None


class ProductTagEntry(BaseModel):
    ProductName: str = Field(..., min_length=1)
    Tags: str = Field(..., min_length=1)


class StoreTypeProfile(BaseModel):
    """Behavioral knobs that make a store type act differently.

    Weight lists are relative (normalized at use time, not here).
    """

    store_type: str = Field(..., min_length=1)
    display_name: str = Field(..., min_length=1)
    basket_lambda: float = Field(..., gt=0, description="Poisson mean items per basket")
    avg_ticket_target: float = Field(..., gt=0, description="Sanity target, USD")
    hourly_weights: list[float] = Field(..., description="24 relative traffic weights")
    daily_weights: list[float] = Field(..., description="7 weights, Monday first")
    monthly_weights: list[float] = Field(..., description="12 seasonality weights, Jan first")
    department_weights: dict[str, float] = Field(..., min_length=1)
    promo_rate: float = Field(..., ge=0, le=1, description="Share of lines with a promotion")
    online_order_share: float = Field(..., ge=0, le=1)
    zones: list[str] = Field(..., min_length=1, description="Store footprint zones for BLE")

    @field_validator("hourly_weights", "daily_weights", "monthly_weights")
    @classmethod
    def _weights_nonneg(cls, v: list[float]) -> list[float]:
        if any(w < 0 for w in v):
            raise ValueError("weight values must be non-negative")
        return v

    @field_validator("hourly_weights")
    @classmethod
    def _24(cls, v: list[float]) -> list[float]:
        if len(v) != 24:
            raise ValueError("hourly_weights must have 24 entries")
        return v

    @field_validator("daily_weights")
    @classmethod
    def _7(cls, v: list[float]) -> list[float]:
        if len(v) != 7:
            raise ValueError("daily_weights must have 7 entries")
        return v

    @field_validator("monthly_weights")
    @classmethod
    def _12(cls, v: list[float]) -> list[float]:
        if len(v) != 12:
            raise ValueError("monthly_weights must have 12 entries")
        return v

# --- retail_setup/dictionaries/loader.py ---
"""Load and validate dictionary JSON sets (shared + one store type)."""

import json
from dataclasses import dataclass, field
from pathlib import Path

from pydantic import BaseModel, ValidationError



@dataclass
class DictionarySet:
    store_type: str
    profile: StoreTypeProfile
    first_names: list[NameEntry] = field(default_factory=list)
    last_names: list[NameEntry] = field(default_factory=list)
    geographies: list[GeographyEntry] = field(default_factory=list)
    tax_rates: list[TaxJurisdictionEntry] = field(default_factory=list)
    products: list[ProductEntry] = field(default_factory=list)
    brands: list[ProductBrandEntry] = field(default_factory=list)
    tags: list[ProductTagEntry] = field(default_factory=list)


def default_dictionary_root() -> Path:
    """Return utility/data/dictionaries, resolved relative to this package.

    Assumes the editable-install src layout where parents[3] == utility/.
    This path resolution breaks under a wheel install (the package is buried
    inside site-packages, not alongside data/).  If you encounter a
    RuntimeError here, pass an explicit ``root`` to ``load_dictionaries``
    instead of relying on this function.
    """
    root = Path(__file__).resolve().parents[3] / "data" / "dictionaries"
    if not root.is_dir():
        raise RuntimeError(
            f"Default dictionary root {root} does not exist. "
            "This function assumes an editable install (src layout) and does not "
            "work under a wheel install. Pass an explicit root to load_dictionaries()."
        )
    return root


def available_store_types(root: Path) -> list[str]:
    if not root.is_dir():
        return []
    return sorted(
        p.name for p in root.iterdir()
        if p.is_dir() and not p.name.startswith("_") and (p / "profile.json").exists()
    )


def load_list(path: Path, model: type[BaseModel]) -> list:
    """Load and validate a JSON array file, returning a list of model instances.

    Raises ValueError with filename and row index on validation failure.
    """
    try:
        raw = json.loads(path.read_text())
    except FileNotFoundError:
        raise ValueError(f"missing dictionary file: {path}") from None
    if not isinstance(raw, list):
        raise ValueError(f"{path.name}: expected a JSON array")
    result = []
    for i, row in enumerate(raw):
        try:
            result.append(model.model_validate(row))
        except ValidationError as exc:
            raise ValueError(f"{path.name}[{i}]: {exc}") from exc
    return result


# Keep the private alias for any callers that haven't migrated yet.
_load_list = load_list


def load_dictionaries(root: Path, store_type: str) -> DictionarySet:
    type_dir = root / store_type
    if not (type_dir / "profile.json").exists():
        raise ValueError(
            f"unknown store type {store_type!r}; available: {available_store_types(root)}"
        )
    shared = root / "_shared"

    try:
        profile = StoreTypeProfile.model_validate(
            json.loads((type_dir / "profile.json").read_text())
        )
    except (json.JSONDecodeError, ValidationError) as exc:
        raise ValueError(f"profile.json: {exc}") from exc
    if profile.store_type != store_type:
        raise ValueError(
            f"profile.json store_type {profile.store_type!r} does not match folder {store_type!r}"
        )

    tags_path = type_dir / "tags.json"
    return DictionarySet(
        store_type=store_type,
        profile=profile,
        first_names=load_list(shared / "first_names.json", NameEntry),
        last_names=load_list(shared / "last_names.json", NameEntry),
        geographies=load_list(shared / "geographies.json", GeographyEntry),
        tax_rates=load_list(shared / "tax_rates.json", TaxJurisdictionEntry),
        products=load_list(type_dir / "products.json", ProductEntry),
        brands=load_list(type_dir / "brands.json", ProductBrandEntry),
        tags=load_list(tags_path, ProductTagEntry) if tags_path.exists() else [],
    )

# --- retail_setup/config/generation.py ---
"""Generation settings (utility/config.yaml). Environment settings live in deploy/config/."""

import calendar
from datetime import date, timedelta
from pathlib import Path

import yaml
from pydantic import BaseModel, Field, model_validator


# Minimum number of synthetic customers, regardless of store_count. The churn
# model (fabric/lakehouse/09-ml-churn-prediction.ipynb) needs at least two
# customers in each class (active / churned) to build a train/test split. With
# geography affinity concentrating purchases on nearby customers, very small
# customer pools (e.g. a 1-store demo deriving 1,000) can leave fewer than two
# churned customers and abort training. Flooring the count keeps small-store
# demos trainable. This is a no-op for store_count >= 5, where the derived
# value (store_count * 1000) already exceeds the floor.
MIN_CUSTOMER_COUNT = 5000


def _subtract_months(anchor: date, months: int) -> date:
    """Return the date ``months`` calendar months before ``anchor`` (day clamped)."""

    month_index = anchor.month - 1 - months
    year = anchor.year + month_index // 12
    month = month_index % 12 + 1
    day = min(anchor.day, calendar.monthrange(year, month)[1])
    return date(year, month, day)


class GenerationConfig(BaseModel):
    store_type: str = "supercenter"
    # Preferred input: number of months of history to generate. The window ends
    # *yesterday* so real-time streaming continues seamlessly from today.
    # ``start_date``/``end_date`` may be provided explicitly instead (back-compat);
    # they are derived from ``months`` only when not both already supplied.
    months: int | None = Field(default=None, ge=1, le=120)
    start_date: date | None = None
    end_date: date | None = None
    store_count: int = Field(default=50, gt=0, le=2000)
    seed: int = 42
    silver_db: str = "silver"
    gold_db: str = "gold"
    # Optional override for dictionary root; when None, default_dictionary_root() is used.
    # Pass an absolute path string to point at a custom dictionary tree (e.g. a Fabric
    # lakehouse Files mount) without mutating the package data directory.
    dictionary_root: str | None = None

    # scale knobs; None -> derived from store_count in the validator below
    dc_count: int | None = Field(default=None, gt=0)
    customer_count: int | None = Field(default=None, gt=0)
    # base in-store transactions per store-day at multiplier 1.0; profiles'
    # hourly/daily/monthly weights shape it, store daily_traffic_multiplier scales it
    transactions_per_store_day: int = Field(default=400, gt=0)
    # fraction of SALE receipts returned per day (Dec 26 spikes 6x, capped 10%)
    # nominal daily return share; Dec 26 applies a 6x spike capped at 0.10, so
    # values near the ceiling flatten the spike - keep this small (~0.01)
    return_rate: float = Field(default=0.01, ge=0.0, le=0.10)
    # network-wide online orders per day at multiplier 1.0; None -> store_count * 8
    online_orders_per_day: int | None = Field(default=None, gt=0)
    # number of category-matched branded SKUs generated per base catalog product
    # (datagen combinatorial SKUs); 1 = one SKU per dictionary row
    brands_per_product: int = Field(default=3, ge=1, le=10)
    # truck load capacity in units; a store-day shipment exceeding this is split
    # across multiple truck legs (datagen multi-truck shipments)
    truck_capacity: int = Field(default=15000, gt=0)

    @model_validator(mode="after")
    def _known_store_type(self) -> "GenerationConfig":
        root = Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()
        known = available_store_types(root)
        if self.store_type not in known:
            raise ValueError(f"store_type {self.store_type!r} not found; available: {known}")
        return self

    @property
    def resolved_dictionary_root(self) -> Path:
        """Resolved dictionary root path (explicit override or package default)."""
        return Path(self.dictionary_root) if self.dictionary_root else default_dictionary_root()

    @model_validator(mode="after")
    def _derive_date_range(self) -> "GenerationConfig":
        """Derive start/end from ``months`` (window ends yesterday) when needed."""

        if self.months is not None and not (self.start_date and self.end_date):
            end = date.today() - timedelta(days=1)
            self.end_date = end
            self.start_date = _subtract_months(end, self.months)
        if self.start_date is None or self.end_date is None:
            raise ValueError("provide `months`, or both `start_date` and `end_date`")
        return self

    @model_validator(mode="after")
    def _date_order(self) -> "GenerationConfig":
        if self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date")
        return self

    @model_validator(mode="after")
    def _derive_scale_defaults(self) -> "GenerationConfig":
        if self.dc_count is None:
            self.dc_count = max(1, self.store_count // 10)
        if self.customer_count is None:
            self.customer_count = self.store_count * 1000
        # Guarantee enough customers for the churn model's two-class train/test
        # split; see MIN_CUSTOMER_COUNT above. Applied to the final value so an
        # explicitly small customer_count is lifted too.
        self.customer_count = max(self.customer_count, MIN_CUSTOMER_COUNT)
        if self.online_orders_per_day is None:
            self.online_orders_per_day = self.store_count * 8
        return self


def load_generation_config(path: Path) -> GenerationConfig:
    return GenerationConfig.model_validate(yaml.safe_load(path.read_text()))

# --- retail_setup/generation/schemas.py ---
"""Authoritative output schemas for generated tables (Plan 2a + 2b scope).

Spark simple type strings. Dimension columns keep the legacy PascalCase names
because the semantic model TMDL binds sourceColumn to them (e.g. StoreNumber,
Cost, MSRP). Most fact tables are snake_case, but several 2b tables have
TMDL-bound PascalCase or mixed-case columns documented per table below.

The TMDL contract test (tests/generation/test_schema_contract.py) is the
arbiter: every sourceColumn in a table's TMDL must be present here with a
compatible type. Extra columns not in TMDL are allowed.

Columns added vs plan (from TMDL audit 2026-06-12):
  fact_receipts: added ("Subtotal", "string") — legacy trace/aggregate string
    column bound as sourceColumn: Subtotal in the semantic model. Not a
    snake_case column; appears to be a pre-existing legacy column the model
    still references.

  --- Plan 2b TMDL deltas (per-table, added to reconcile TMDL bindings) ---

  fact_ble_pings:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId (PascalCase),
      not customer_id. Plan listed customer_id (kept as extra); TMDL column added.
    - ("__index_level_0__", "long") — TMDL-bound legacy pandas-index column.

  fact_customer_zone_changes:
    - TMDL uses PascalCase for all data columns: StoreID, CustomerBLEId,
      FromZone, ToZone (plan listed snake_case equivalents, kept as extras).
    - ("StoreID", "long"), ("CustomerBLEId", "string"), ("FromZone", "string"),
      ("ToZone", "string") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_marketing:
    - ("CustomerId", "double") — TMDL sourceColumn is CustomerId; plan's
      customer_id kept as extra.
    - ("CostCents", "long") — TMDL sourceColumn is CostCents; plan's cost_cents
      kept as extra.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_promo_lines:
    - TMDL uses PascalCase: ReceiptId, PromoCode, LineNumber, ProductID, Qty,
      DiscountAmount, DiscountCents (plan listed snake_case equivalents, kept
      as extras).
    - ("ReceiptId", "string"), ("PromoCode", "string"), ("LineNumber", "long"),
      ("ProductID", "long"), ("Qty", "long"), ("DiscountAmount", "string"),
      ("DiscountCents", "long") — TMDL-bound columns added.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_online_order_lines:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_reorders:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_moves:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_truck_inventory:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_dc_inventory_txn:
    - ("Source", "string") — TMDL sourceColumn is PascalCase Source. The
      lowercase ``source`` that appears during construction is renamed to
      ``Source`` in the final select (inventory.py); keeping both would cause
      a case-insensitive collision that Delta on Fabric rejects.
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_store_inventory_txn:
    - ("__index_level_0__", "long") — TMDL-bound.

  fact_stockouts:
    - ("__index_level_0__", "long") — TMDL-bound.

Columns in plan but NOT in TMDL (allowed — Direct Lake ignores extra columns):
  fact_receipts: trace_id, tender_type
  fact_payments: tender_type (not in payments TMDL at all)
  fact_ble_pings: customer_id (TMDL uses CustomerId), trace_id, event_ts,
    event_date
  fact_customer_zone_changes: store_id, customer_ble_id, from_zone, to_zone,
    event_ts, trace_id, event_date
  fact_marketing: customer_id, cost_cents, event_ts, trace_id, event_date,
    channel (extra beyond TMDL-bound channel — already in TMDL, fine)
  fact_promo_lines: receipt_id_ext, promo_code, line_number, product_id,
    quantity, discount_amount, discount_cents, event_ts, trace_id, event_date
  fact_dc_inventory_txn: trace_id, event_ts, event_date
    (lowercase source is renamed to Source in the final select; not an extra)
"""

# table -> list of (column, spark_type)
TABLES: dict[str, list[tuple[str, str]]] = {
    "dim_geographies": [
        ("ID", "long"), ("City", "string"), ("State", "string"), ("ZipCode", "string"),
        ("District", "string"), ("Region", "string"),
    ],
    "dim_stores": [
        ("ID", "long"), ("StoreNumber", "string"), ("Address", "string"),
        ("GeographyID", "long"), ("tax_rate", "double"), ("volume_class", "string"),
        ("store_format", "string"), ("operating_hours", "string"),
        ("daily_traffic_multiplier", "double"),
    ],
    "dim_distribution_centers": [
        ("ID", "long"), ("DCNumber", "string"), ("Address", "string"),
        ("GeographyID", "long"),
    ],
    "dim_trucks": [
        ("ID", "long"), ("LicensePlate", "string"), ("Refrigeration", "boolean"),
        # double, not long: NULL for pool trucks + Direct Lake nullability
        ("DCID", "double"),
    ],
    "dim_customers": [
        ("ID", "long"), ("FirstName", "string"), ("LastName", "string"),
        ("Address", "string"), ("GeographyID", "long"), ("LoyaltyCard", "string"),
        ("Phone", "string"), ("BLEId", "string"), ("AdId", "string"),
    ],
    "dim_products": [
        ("ID", "long"), ("ProductName", "string"), ("Brand", "string"),
        ("Company", "string"), ("Department", "string"), ("Category", "string"),
        ("Subcategory", "string"), ("Cost", "double"), ("MSRP", "double"),
        ("SalePrice", "double"), ("RequiresRefrigeration", "boolean"),
        ("LaunchDate", "timestamp"), ("taxability", "string"), ("Tags", "string"),
    ],
    "dim_date": [
        ("date_key", "long"), ("date", "date"), ("year", "long"), ("quarter", "long"),
        ("month", "long"), ("month_name", "string"), ("day", "long"),
        ("day_of_week", "long"), ("day_name", "string"), ("week_of_year", "long"),
        ("is_weekend", "long"), ("fiscal_year", "long"), ("fiscal_quarter", "long"),
    ],
    "fact_receipts": [
        ("receipt_id_ext", "string"), ("trace_id", "string"), ("event_ts", "timestamp"),
        ("event_date", "date"), ("store_id", "long"), ("customer_id", "long"),
        ("receipt_type", "string"), ("tender_type", "string"),
        ("subtotal_cents", "long"), ("discount_amount", "string"),
        ("tax_cents", "long"), ("total_cents", "long"),
        ("subtotal_amount", "string"), ("tax_amount", "string"),
        ("total_amount", "string"), ("payment_method", "string"),
        # Legacy column bound by the semantic model (sourceColumn: Subtotal).
        # Present in fact_receipts.tmdl as a string column; added here to
        # satisfy the TMDL contract test (TMDL arbiter rule).
        ("Subtotal", "string"),
    ],
    "fact_receipt_lines": [
        ("receipt_id_ext", "string"), ("event_ts", "timestamp"), ("event_date", "date"),
        ("line_num", "int"), ("product_id", "long"), ("quantity", "int"),
        ("unit_price", "string"), ("unit_cents", "long"),
        ("ext_price", "string"), ("ext_cents", "long"), ("promo_code", "string"),
    ],
    "fact_payments": [
        ("receipt_id_ext", "string"), ("order_id_ext", "string"),
        ("event_ts", "timestamp"), ("event_date", "date"),
        ("payment_method", "string"), ("amount_cents", "long"), ("amount", "string"),
        ("transaction_id", "string"), ("status", "string"),
        ("decline_reason", "string"), ("processing_time_ms", "long"),
        ("store_id", "long"), ("customer_id", "long"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2b: 15 remaining fact tables
    # -----------------------------------------------------------------------
    "fact_store_ops": [
        # TMDL-bound: trace_id, store_id (int64→long), operation_type, event_date
        # Extra (not in TMDL, allowed): event_ts
        ("event_ts", "timestamp"), ("trace_id", "string"), ("store_id", "long"),
        ("operation_type", "string"), ("event_date", "date"),
    ],
    "fact_foot_traffic": [
        # TMDL-bound: count (int64→long), zone, dwell_seconds (int64→long),
        #   store_id (int64→long), sensor_id, event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"), ("trace_id", "string"), ("store_id", "long"),
        ("sensor_id", "string"), ("zone", "string"), ("dwell_seconds", "long"),
        ("count", "long"), ("event_date", "date"),
    ],
    "fact_ble_pings": [
        # TMDL-bound: beacon_id, rssi (int64→long), customer_ble_id, zone,
        #   store_id (int64→long), CustomerId (double), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId)
        ("event_ts", "timestamp"), ("trace_id", "string"), ("store_id", "long"),
        ("beacon_id", "string"), ("customer_ble_id", "string"),
        ("customer_id", "double"),
        # TMDL-bound PascalCase column (sourceColumn: CustomerId)
        ("CustomerId", "double"),
        ("rssi", "long"), ("zone", "string"),
        ("event_date", "date"),
        # TMDL-bound legacy pandas-index column
        ("__index_level_0__", "long"),
    ],
    "fact_customer_zone_changes": [
        # TMDL-bound PascalCase: StoreID (int64→long), CustomerBLEId (string),
        #   FromZone (string), ToZone (string), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   store_id, customer_ble_id, from_zone, to_zone (snake_case plan names)
        ("event_ts", "timestamp"), ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("store_id", "long"), ("customer_ble_id", "string"),
        ("from_zone", "string"), ("to_zone", "string"),
        # TMDL-bound PascalCase columns
        ("StoreID", "long"), ("CustomerBLEId", "string"),
        ("FromZone", "string"), ("ToZone", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_marketing": [
        # TMDL-bound: device, cost, customer_ad_id, impression_id_ext, campaign_id,
        #   creative_id, channel, CustomerId (double), CostCents (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   customer_id (plan name; TMDL uses CustomerId),
        #   cost_cents (plan name; TMDL uses CostCents)
        ("event_ts", "timestamp"), ("trace_id", "string"), ("channel", "string"),
        ("campaign_id", "string"), ("creative_id", "string"),
        ("customer_ad_id", "string"),
        # snake_case plan columns (extras)
        ("customer_id", "double"), ("cost_cents", "long"),
        # TMDL-bound PascalCase columns (sourceColumn: CustomerId, CostCents)
        ("CustomerId", "double"), ("CostCents", "long"),
        ("impression_id_ext", "string"), ("cost", "string"),
        ("device", "string"), ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_promotions": [
        # TMDL-bound: receipt_id_ext, promo_code, discount_amount, discount_cents
        #   (int64→long), discount_type, product_count (int64→long), product_ids,
        #   store_id (int64→long), customer_id (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts, trace_id
        ("event_ts", "timestamp"), ("trace_id", "string"), ("receipt_id_ext", "string"),
        ("promo_code", "string"), ("discount_amount", "string"),
        ("discount_cents", "long"), ("discount_type", "string"),
        ("product_count", "long"), ("product_ids", "string"), ("store_id", "long"),
        ("customer_id", "long"), ("event_date", "date"),
    ],
    "fact_promo_lines": [
        # TMDL-bound PascalCase: ReceiptId, PromoCode, LineNumber (int64→long),
        #   ProductID (int64→long), Qty (int64→long), DiscountAmount (string),
        #   DiscountCents (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   receipt_id_ext, promo_code, line_number, product_id, quantity,
        #   discount_amount, discount_cents (snake_case plan names)
        ("event_ts", "timestamp"), ("trace_id", "string"),
        # snake_case plan columns (extras)
        ("receipt_id_ext", "string"), ("promo_code", "string"),
        ("line_number", "long"), ("product_id", "long"),
        ("quantity", "long"), ("discount_amount", "string"),
        ("discount_cents", "long"),
        # TMDL-bound PascalCase columns
        ("ReceiptId", "string"), ("PromoCode", "string"),
        ("LineNumber", "long"), ("ProductID", "long"),
        ("Qty", "long"), ("DiscountAmount", "string"), ("DiscountCents", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_online_order_headers": [
        # TMDL-bound: customer_id (int64→long), subtotal_cents (int64→long),
        #   order_id_ext, payment_method, total_amount, total_cents (int64→long),
        #   tax_amount, subtotal_amount, tax_cents (int64→long), event_date
        # Extra (not in TMDL, allowed): event_ts
        ("order_id_ext", "string"), ("customer_id", "long"),
        ("subtotal_cents", "long"), ("tax_cents", "long"), ("total_cents", "long"),
        ("subtotal_amount", "string"), ("tax_amount", "string"),
        ("total_amount", "string"), ("payment_method", "string"),
        ("event_ts", "timestamp"), ("event_date", "date"),
    ],
    "fact_online_order_lines": [
        # TMDL-bound: order_id, unit_cents (int64→long), fulfillment_mode,
        #   promo_code, node_type, product_id (int64→long), ext_price, ext_cents
        #   (int64→long), line_num (int64→long), fulfillment_status, unit_price,
        #   quantity (int64→long), node_id (int64→long), __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): picked_ts, shipped_ts, delivered_ts,
        #   event_ts, event_date
        ("order_id", "string"), ("product_id", "long"), ("line_num", "long"),
        ("quantity", "long"), ("unit_price", "string"), ("unit_cents", "long"),
        ("ext_price", "string"), ("ext_cents", "long"), ("promo_code", "string"),
        ("fulfillment_mode", "string"), ("fulfillment_status", "string"),
        ("node_type", "string"), ("node_id", "long"),
        ("picked_ts", "timestamp"), ("shipped_ts", "timestamp"),
        ("delivered_ts", "timestamp"), ("event_ts", "timestamp"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_reorders": [
        # TMDL-bound: store_id (int64→long), dc_id (int64→long), product_id
        #   (int64→long), current_quantity (int64→long), reorder_quantity
        #   (int64→long), reorder_point (int64→long), priority,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"), ("trace_id", "string"), ("store_id", "long"),
        ("dc_id", "long"), ("product_id", "long"), ("current_quantity", "long"),
        ("reorder_quantity", "long"), ("reorder_point", "long"), ("priority", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_moves": [
        # TMDL-bound: status, shipment_id, dc_id (int64→long), truck_id
        #   (int64→long), store_id (int64→long), actual_unload_duration (double),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, eta, etd,
        #   departure_time, event_date
        ("event_ts", "timestamp"), ("trace_id", "string"), ("truck_id", "long"),
        ("dc_id", "long"), ("store_id", "long"), ("shipment_id", "string"),
        ("status", "string"), ("eta", "timestamp"), ("etd", "timestamp"),
        ("departure_time", "timestamp"), ("actual_unload_duration", "double"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_truck_inventory": [
        # TMDL-bound: truck_id (int64→long), shipment_id, product_id (int64→long),
        #   quantity (int64→long), action, location_id (int64→long), location_type,
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"), ("trace_id", "string"), ("truck_id", "long"),
        ("shipment_id", "string"), ("product_id", "long"), ("quantity", "long"),
        ("action", "string"), ("location_id", "long"), ("location_type", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_dc_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), dc_id (int64→long),
        #   balance (int64→long), product_id (int64→long), Source (PascalCase!),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date,
        #   source (lowercase snake_case plan name)
        ("event_ts", "timestamp"), ("trace_id", "string"), ("dc_id", "long"),
        ("product_id", "long"), ("quantity", "long"), ("balance", "long"),
        ("txn_type", "string"),
        # TMDL-bound PascalCase column (sourceColumn: Source). The lowercase
        # ``source`` used during construction is renamed AS "Source" in
        # inventory.py before the final select; keeping both would produce a
        # case-insensitive duplicate that Delta on Fabric rejects.
        ("Source", "string"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_store_inventory_txn": [
        # TMDL-bound: txn_type, quantity (int64→long), balance (int64→long),
        #   product_id (int64→long), store_id (int64→long), source (lowercase OK),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"), ("trace_id", "string"), ("store_id", "long"),
        ("product_id", "long"), ("quantity", "long"), ("balance", "long"),
        ("txn_type", "string"), ("source", "string"), ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    "fact_stockouts": [
        # PascalCase IS the contract: TMDL binds StoreID (double), DCID (double),
        #   ProductID (int64→long), LastKnownQuantity (int64→long),
        #   __index_level_0__ (int64→long)
        # Extra (not in TMDL, allowed): event_ts, trace_id, event_date
        ("event_ts", "timestamp"), ("trace_id", "string"),
        ("StoreID", "double"), ("DCID", "double"),
        ("ProductID", "long"), ("LastKnownQuantity", "long"),
        ("event_date", "date"),
        ("__index_level_0__", "long"),
    ],
    # -----------------------------------------------------------------------
    # Plan 2c: 9 Gold (gold) aggregate tables
    # TMDL audit 2026-06-12: bindings match exactly; `computed_at` and `as_of`
    # are produced by the legacy transforms but unbound in TMDL (extras OK).
    # TMDL `day`/`ts` are dateTime — TYPE_COMPAT accepts timestamp|date.
    # -----------------------------------------------------------------------
    "sales_minute_store": [
        ("store_id", "long"), ("ts", "timestamp"), ("total_sales", "double"),
        ("receipts", "long"), ("avg_basket", "double"),
    ],
    "top_products_15m": [
        ("product_id", "long"), ("revenue", "double"), ("units", "long"),
        ("computed_at", "timestamp"),  # produced by legacy code, unbound in TMDL
    ],
    "inventory_position_current": [
        ("store_id", "long"), ("product_id", "long"), ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "dc_inventory_position_current": [
        ("dc_id", "long"), ("product_id", "long"), ("on_hand", "long"),
        ("as_of", "timestamp"),
    ],
    "truck_dwell_daily": [
        ("site", "string"), ("day", "date"), ("avg_dwell_min", "double"),
        ("trucks", "long"),
    ],
    "online_sales_daily": [
        ("day", "date"), ("orders", "long"), ("subtotal", "double"),
        ("tax", "double"), ("total", "double"), ("avg_order_value", "double"),
    ],
    "zone_dwell_minute": [
        ("store_id", "long"), ("zone", "string"), ("ts", "timestamp"),
        ("avg_dwell", "double"), ("customers", "long"),
    ],
    "marketing_cost_daily": [
        ("campaign_id", "string"), ("day", "date"), ("impressions", "long"),
        ("cost", "double"),
    ],
    "tender_mix_daily": [
        ("day", "date"), ("payment_method", "string"), ("transactions", "long"),
        ("total_amount", "double"),
    ],
}


_SPARK_TYPE_MAP = None


def _type_map():
    """Lazy-import PySpark type map (avoids import cost when pyspark not installed)."""
    global _SPARK_TYPE_MAP
    if _SPARK_TYPE_MAP is None:
        from pyspark.sql.types import (
            BooleanType, DateType, DoubleType, IntegerType,
            LongType, StringType, TimestampType,
        )
        _SPARK_TYPE_MAP = {
            "long": LongType(),
            "int": IntegerType(),
            "string": StringType(),
            "double": DoubleType(),
            "boolean": BooleanType(),
            "timestamp": TimestampType(),
            "date": DateType(),
        }
    return _SPARK_TYPE_MAP


def spark_schema(table: str):
    """Build a StructType for createDataFrame with explicit types."""
    from pyspark.sql.types import StructField, StructType

    tmap = _type_map()
    fields = [
        StructField(name, tmap[typ], nullable=True)
        for name, typ in TABLES[table]
    ]
    return StructType(fields)


def column_names(table: str) -> list[str]:
    return [name for name, _ in TABLES[table]]

# --- retail_setup/generation/runtime.py ---
"""Deterministic seeding + partition grids for the generation engine."""

import hashlib
from datetime import date, timedelta

from pyspark.sql import DataFrame, SparkSession


class seeded_draws:
    """Deterministic draw expressions bound to a seed.

    u(cols, salt)        -> uniform [0,1) double
    gauss(cols, salt)    -> ~N(0,1) (Irwin-Hall of 3 uniforms, scaled)
    h64(cols, salt)      -> non-negative long hash
    pick_by_weights(cols, salt, [(value, weight), ...]) -> weighted categorical
    """

    _U_MOD = 10**12

    def __init__(self, seed: int):
        self.seed = seed

    def h64(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return F.pmod(F.xxhash64(*cols, F.lit(f"{salt}|{self.seed}")), F.lit(2**62))

    def u(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        return (self.h64(cols, salt) % F.lit(self._U_MOD)) / F.lit(float(self._U_MOD))

    def gauss(self, cols: list, salt: str):
        from pyspark.sql import functions as F

        s = self.u(cols, f"{salt}|g1") + self.u(cols, f"{salt}|g2") + self.u(cols, f"{salt}|g3")
        return (s - F.lit(1.5)) * F.lit(2.0)

    def pick_by_weights(self, cols: list, salt: str, weighted: list[tuple[str, float]]):
        from pyspark.sql import functions as F

        total = sum(w for _, w in weighted)
        uu = self.u(cols, salt)
        expr, acc = None, 0.0
        for value, w in weighted[:-1]:
            acc += w / total
            expr = expr.when(uu < acc, value) if expr is not None else F.when(uu < acc, value)
        return expr.otherwise(weighted[-1][0]) if expr is not None else F.lit(weighted[0][0])


def derive_seed(global_seed: int, section: str, key: int, day: date) -> int:
    """Stable 31-bit seed from (global_seed, section, key, day).

    Independent of Spark partitioning/execution order — safe for per-row
    or per-group RNG seeding.
    """
    payload = f"{global_seed}|{section}|{key}|{day.isoformat()}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") % (2**31)


def legacy_index(*key_cols):
    """Deterministic long for the legacy __index_level_0__ pandas column.

    The semantic model binds the column by name only; nothing consumes its
    values, so a hash beats a dense row_number — which forced a
    single-partition global sort at full volume.
    """
    from pyspark.sql import functions as F

    return F.pmod(F.xxhash64(*key_cols, F.lit("__legacy_index__")), F.lit(2**62))


def store_day_grid(
    spark: SparkSession,
    store_ids: list[int],
    start: date,
    end: date,
    global_seed: int,
    section: str,
) -> DataFrame:
    """store_id x day cross grid.

    Rows are computed driver-side so the engine never depends on F.rand()'s
    partition-arrangement semantics for keys. Fine for realistic grids
    (hundreds of stores x a year ~ 10^5 rows collected on the driver).

    Spark-native generators derive their own draws via seeded_draws/xxhash64
    and do not need a driver-side seed column.
    """
    day_list = [start + timedelta(days=i) for i in range((end - start).days + 1)]
    rows = [
        (s, d)
        for s in store_ids
        for d in day_list
    ]
    # the rows ARE the full grid — no crossJoin/join round-trip needed
    return spark.createDataFrame(rows, "store_id long, day date")

# --- retail_setup/generation/dims.py ---
"""Dimension generation: driver-side numpy/pandas -> Spark DataFrames.

Semantics ported from datagen master_generators (ID schemes, tax lookup,
pricing rules); column names/types from schemas.TABLES, which the TMDL
contract test guards.
"""

from datetime import date, datetime, timedelta

import numpy as np
from pyspark.sql import DataFrame, SparkSession


# datagen StoreProfiler equivalents (volume class -> traffic multiplier range).
# Five tiers incl. KIOSK, matching datagen's StoreVolumeClass (the kiosk tier
# was dropped in the first port; restored here for store-volume variety).
VOLUME_CLASSES = [
    ("flagship", 0.05, (1.8, 2.5)),
    ("high_volume", 0.15, (1.3, 1.8)),
    ("standard", 0.55, (0.8, 1.3)),
    ("low_volume", 0.20, (0.4, 0.8)),
    ("kiosk", 0.05, (0.25, 0.35)),
]
# Five formats incl. EXPRESS (datagen's StoreFormat.EXPRESS). store_activity
# carries a matching express zone-share row.
STORE_FORMATS = ["hypermarket", "superstore", "standard", "neighborhood", "express"]
OPERATING_HOURS = ["6-22", "7-22", "7-23", "24h"]
DEFAULT_TAX_RATE = 0.07407  # datagen receipts_mixin fallback
REFRIGERATED_CATEGORIES = {"Produce", "Dairy & Eggs", "Dairy & Alternatives",
                           "Meat & Poultry", "Meat & Seafood", "Seafood", "Frozen"}

# Share of customers placed in a geography that has a store (the rest are
# scattered anywhere). Restores datagen's "customers live near stores" locality
# so receipts can draw a same-geography customer most of the time.
CUSTOMER_HOME_AFFINITY = 0.70

# Map a product department/category token to a brand-dictionary Category so a
# product's brand fits its department (datagen product_generator parity: a
# grocery item never carries a hardware brand). Tokens are matched lowercased;
# substring matches (e.g. "home" in "home & garden") are tried before falling
# back to the whole brand pool.
_BRAND_CAT_SYNONYMS = {
    "grocery": "food", "fresh": "food", "pantry": "food", "snacks": "food",
    "beverages": "food", "health & beauty": "health", "home & garden": "home",
    "office supplies": "office", "pet supplies": "pet",
    "sports & recreation": "sports", "apparel": "clothing",
}


def _match_brand_category(key: str, brands_by_cat: dict[str, list]) -> list | None:
    """Resolve a product department/category to a brand pool, or None."""
    k = key.strip().lower()
    if k in brands_by_cat:
        return brands_by_cat[k]
    syn = _BRAND_CAT_SYNONYMS.get(k)
    if syn and syn in brands_by_cat:
        return brands_by_cat[syn]
    for cat, pool in brands_by_cat.items():
        if cat and (cat in k or k in cat):
            return pool
    return None


def compute_pricing(base_price: float, rng: np.random.Generator) -> tuple[float, float, float]:
    """Return ``(cost, msrp, sale_price)`` for a base price.

    Ported from datagen ``PricingCalculator`` (shared/validators/pricing.py):

    - MSRP = BasePrice +/-15%
    - SalePrice = MSRP (60% of the time) OR MSRP discounted 5-35% (40%)
    - Cost = 50-85% of SalePrice

    Always guarantees ``Cost < SalePrice <= MSRP``.
    """
    msrp = max(0.01, round(base_price * (1.0 + rng.uniform(-0.15, 0.15)), 2))
    if rng.random() < 0.60:
        sale = msrp
    else:
        sale = max(0.01, round(msrp * (1.0 - rng.uniform(0.05, 0.35)), 2))
    cost = sale * rng.uniform(0.50, 0.85)
    cost = min(cost, sale - 0.01)
    cost = max(0.01, round(cost, 2))
    return float(cost), float(msrp), float(sale)


def _addr(rng: np.random.Generator) -> str:
    return (
        f"{int(rng.integers(100, 9999))} "
        f"{str(rng.choice(['Main', 'Oak', 'Maple', 'Market', 'Commerce', 'Liberty']))} "
        f"{str(rng.choice(['St', 'Ave', 'Blvd', 'Rd']))}"
    )


def generate_dimensions(
    spark: SparkSession, dicts: DictionarySet, cfg: GenerationConfig
) -> dict[str, DataFrame]:
    rng = np.random.default_rng(derive_seed(cfg.seed, "dims", 0, cfg.start_date))
    out: dict[str, DataFrame] = {}

    # --- geographies: sample from dictionary, sequential IDs
    n_geo = min(len(dicts.geographies), max(cfg.store_count * 2, cfg.dc_count * 2, 20))
    geo_idx = rng.choice(len(dicts.geographies), size=n_geo, replace=False)
    geos = [dicts.geographies[i] for i in geo_idx]
    geo_rows = [
        (i + 1, g.City, g.State, g.Zip, g.District, g.Region)
        for i, g in enumerate(geos)
    ]
    out["dim_geographies"] = spark.createDataFrame(geo_rows, spark_schema("dim_geographies"))

    # tax lookup hierarchy (datagen TaxCalculator parity):
    #   exact (State, City) -> (State, County) average -> State average -> default.
    # County is resolved from the tax dictionary; geographies carry no county of
    # their own, so the county tier only applies to cities present in the dict.
    city_acc: dict[tuple[str, str], list[float]] = {}
    city_county: dict[tuple[str, str], str] = {}
    county_acc: dict[tuple[str, str], list[float]] = {}
    state_rates: dict[str, list[float]] = {}
    for t in dicts.tax_rates:
        rate = float(t.CombinedRate)
        city_acc.setdefault((t.StateCode, t.City), []).append(rate)
        city_county[(t.StateCode, t.City)] = t.County
        county_acc.setdefault((t.StateCode, t.County), []).append(rate)
        state_rates.setdefault(t.StateCode, []).append(rate)
    by_city = {k: float(np.mean(v)) for k, v in city_acc.items()}
    by_county = {k: float(np.mean(v)) for k, v in county_acc.items()}

    def tax_for(state: str, city: str) -> float:
        if (state, city) in by_city:
            return by_city[(state, city)]
        county = city_county.get((state, city))
        if county is not None and (state, county) in by_county:
            return by_county[(state, county)]
        if state in state_rates:
            return float(np.mean(state_rates[state]))
        return DEFAULT_TAX_RATE

    # --- DCs first (datagen: stores constrained to DC states)
    dc_geo_idx = rng.choice(n_geo, size=cfg.dc_count, replace=cfg.dc_count > n_geo)
    dc_rows = [
        (i + 1, f"DC{i + 1:03d}", _addr(rng), int(dc_geo_idx[i]) + 1)
        for i in range(cfg.dc_count)
    ]
    out["dim_distribution_centers"] = spark.createDataFrame(
        dc_rows, spark_schema("dim_distribution_centers")
    )

    # --- stores in DC states
    dc_states = {geos[int(g)].State for g in dc_geo_idx}
    eligible = [i for i, g in enumerate(geos) if g.State in dc_states] or list(range(n_geo))
    store_rows = []
    store_geo_indices: list[int] = []
    for sid in range(1, cfg.store_count + 1):
        gi = int(rng.choice(eligible))
        store_geo_indices.append(gi)
        g = geos[gi]
        classes, probs = zip(*[(c, p) for c, p, _ in VOLUME_CLASSES])
        vc = str(rng.choice(classes, p=probs))
        lo, hi = next(r for c, _, r in VOLUME_CLASSES if c == vc)
        store_rows.append((
            sid,
            f"S{sid:06d}",
            _addr(rng),
            gi + 1,
            tax_for(g.State, g.City),
            vc,
            str(rng.choice(STORE_FORMATS)),
            str(rng.choice(OPERATING_HOURS)),
            float(np.round(rng.uniform(lo, hi), 2)),
        ))
    out["dim_stores"] = spark.createDataFrame(store_rows, spark_schema("dim_stores"))

    # --- trucks: 85% assigned round-robin to DCs, 15% pool (DCID NULL); 40% refrigerated
    n_trucks = max(cfg.dc_count * 3, 6)
    n_assigned = int(round(n_trucks * 0.85))
    truck_rows = []
    for tid in range(1, n_trucks + 1):
        plate = f"TRK{tid:04d}{chr(65 + tid % 26)}"
        refrig = bool(rng.random() < 0.4)
        dcid = float((tid - 1) % cfg.dc_count + 1) if tid <= n_assigned else None
        truck_rows.append((tid, plate, refrig, dcid))
    out["dim_trucks"] = spark.createDataFrame(truck_rows, spark_schema("dim_trucks"))

    # --- customers; ~70% placed in a store's geography (datagen home-store
    #     locality) so receipts can resolve a same-geography "local" shopper.
    first = [n.Name for n in dicts.first_names]
    last = [n.Name for n in dicts.last_names]
    cust_rows = []
    for cid in range(1, cfg.customer_count + 1):
        if store_geo_indices and rng.random() < CUSTOMER_HOME_AFFINITY:
            gi = int(rng.choice(store_geo_indices))
        else:
            gi = int(rng.integers(0, n_geo))
        cust_rows.append((
            cid,
            str(rng.choice(first)),
            str(rng.choice(last)),
            _addr(rng),
            gi + 1,
            f"LC{cid:06d}{int(rng.integers(0, 1000)):03d}",
            f"555-{int(rng.integers(200, 999))}-{int(rng.integers(1000, 9999))}",
            "BLE" + np.base_repr(cid, 36).rjust(6, "0"),
            f"AD{cid:08d}",
        ))
    out["dim_customers"] = spark.createDataFrame(cust_rows, spark_schema("dim_customers"))

    # --- products: each base product is offered by up to brands_per_product
    #     category-matched brands (datagen combinatorial SKUs). Pricing/launch
    #     are re-rolled per branded variant for realistic price spread.
    brands_by_cat: dict[str, list] = {}
    for b in dicts.brands:
        brands_by_cat.setdefault(b.Category.strip().lower(), []).append(b)
    all_brands = list(dicts.brands)
    tags_by_product = {t.ProductName: t.Tags for t in dicts.tags}
    # Use naive UTC datetimes — Spark session timezone is UTC (set in conftest fixture)
    hist_start = datetime.combine(cfg.start_date, datetime.min.time())
    prod_rows = []
    pid = 0
    for p in dicts.products:
        pool = (_match_brand_category(p.Department, brands_by_cat)
                or _match_brand_category(p.Category, brands_by_cat)
                or all_brands)
        k = min(cfg.brands_per_product, len(pool))
        brand_idx = rng.choice(len(pool), size=k, replace=False)
        taxability = (
            "NON_TAXABLE" if p.Department in {"Fresh", "Grocery"} and "Candy" not in p.Category
            else "REDUCED_RATE" if p.Department in {"Clothing", "Apparel"}
            else "TAXABLE"
        )
        refrigerated = bool(p.Category in REFRIGERATED_CATEGORIES)
        tags = p.Tags or tags_by_product.get(p.ProductName)
        for j in brand_idx:
            pid += 1
            chosen = pool[int(j)]
            launch_r = float(rng.random())  # 60% before history, 30% first half, 10% later
            if launch_r < 0.6:
                launch = hist_start - timedelta(days=int(rng.integers(30, 1500)))
            elif launch_r < 0.9:
                launch = hist_start + timedelta(days=int(rng.integers(0, 183)))
            else:
                launch = hist_start + timedelta(days=int(rng.integers(183, 366)))
            cost, msrp, sale = compute_pricing(float(p.BasePrice), rng)
            prod_rows.append((
                pid,
                p.ProductName,
                chosen.Brand,
                chosen.Company,
                p.Department,
                p.Category,
                p.Subcategory,
                cost,
                msrp,
                sale,
                refrigerated,
                launch,
                taxability,
                tags,
            ))
    out["dim_products"] = spark.createDataFrame(prod_rows, spark_schema("dim_products"))
    return out


def generate_dim_date(spark: SparkSession, start: date, end: date) -> DataFrame:
    """Exact port of 02-historical-data-load's dim_date (fiscal year starts July)."""
    rows = []
    d = start
    while d <= end:
        rows.append((
            int(d.strftime("%Y%m%d")),
            d,
            d.year,
            (d.month - 1) // 3 + 1,
            d.month,
            d.strftime("%B"),
            d.day,
            d.isoweekday(),
            d.strftime("%A"),
            int(d.strftime("%U")),
            1 if d.isoweekday() >= 6 else 0,
            d.year if d.month >= 7 else d.year - 1,
            ((d.month - 7) % 12) // 3 + 1,
        ))
        d += timedelta(days=1)
    return spark.createDataFrame(rows, spark_schema("dim_date"))

# --- retail_setup/generation/receipts.py ---
"""Receipts fact group, Spark-native.

Randomness: every stochastic decision derives a uniform double from
xxhash64(key columns, salt) via `runtime.seeded_draws` — partition-
arrangement-independent, so output is deterministic for a (config, seed) pair
regardless of cluster shape. The seed is folded into the salt delimiter inside
`seeded_draws`, keeping generators decoupled from one another.

Count distributions (store-day receipt counts, basket sizes) use a clamped
normal approximation of Poisson — `max(1, round(N(lambda, sqrt(lambda))))` —
a documented deviation from datagen's per-row RNG; statistically equivalent
at demo scale and fully vectorizable.

Money is integer cents end-to-end; tax replicates datagen's basis-point
integer formula exactly:
    rate_bps = round(rate * 10000); mult = 100/50/0 by taxability;
    tax = (ext_cents * rate_bps * mult + 500_000) // 1_000_000
implemented with Spark integer `DIV` so no float rounding is involved.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# (method, mix weight, decline multiplier, processing_ms lo, processing_ms hi)
TENDERS = [
    ("CREDIT_CARD", 0.4, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.3, 0.8, 1200, 3500),
    ("CASH", 0.2, 0.0, 500, 2000),
    ("MOBILE_PAY", 0.1, 1.2, 800, 2500),
]
BASE_DECLINE = 0.025
DECLINE_REASONS = [
    "INSUFFICIENT_FUNDS", "CARD_EXPIRED", "INVALID_CVV", "NETWORK_ERROR",
    "FRAUD_SUSPECTED", "CARD_BLOCKED", "LIMIT_EXCEEDED",
]

# Share of in-store receipts whose customer lives in the store's geography
# (datagen home-store locality). The rest draw a network-wide customer.
GEO_AFFINITY = 0.70

# Per-department seasonal demand multipliers by month (Jan..Dec), keyed by a
# lowercase token matched as a substring of the department name so the one
# table works across store types (datagen SeasonalPatterns category effects).
SEASONAL_LIFT: dict[str, list[float]] = {
    "electronics": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 2.5, 2.0],
    "seasonal":    [0.6, 0.7, 0.9, 1.0, 1.3, 1.5, 1.5, 1.2, 1.0, 1.4, 2.0, 2.5],
    "garden":      [0.5, 0.6, 1.3, 1.8, 2.0, 1.8, 1.4, 1.0, 0.8, 0.6, 0.5, 0.5],
    "home":        [0.8, 0.8, 1.2, 1.5, 1.6, 1.4, 1.2, 1.0, 0.9, 0.8, 1.0, 1.1],
    "sport":       [0.9, 0.9, 1.1, 1.3, 1.5, 1.6, 1.6, 1.3, 1.1, 1.0, 1.1, 1.2],
    "clothing":    [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "apparel":     [1.0, 0.9, 1.1, 1.1, 1.0, 1.0, 1.0, 1.4, 1.2, 1.0, 1.3, 1.5],
    "office":      [1.1, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.6, 1.4, 1.0, 1.0, 1.1],
    "toys":        [0.8, 0.8, 0.8, 0.9, 0.9, 1.0, 1.0, 1.0, 1.0, 1.1, 2.0, 3.0],
    "baby":        [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.1, 1.2],
    "grocery":     [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.3, 1.4],
    "fresh":       [1.0, 1.0, 1.0, 1.0, 1.05, 1.1, 1.1, 1.05, 1.0, 1.05, 1.3, 1.4],
    "health":      [1.3, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.1],
    "beauty":      [1.2, 1.05, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.05, 1.2, 1.5],
    "automotive":  [1.0, 1.0, 1.05, 1.1, 1.2, 1.2, 1.2, 1.1, 1.0, 1.0, 1.0, 1.0],
}

# Named promo catalog (datagen promotion_utils parity): seasonal codes apply
# only in their eligible months; evergreen SAVE/CLEARANCE codes fill in
# otherwise. Some codes carry a minimum-purchase threshold; BOGO is a qty-based
# buy-one-get-one. (code, discount_pct, eligible_months | None, min_cents, kind)
PROMO_CATALOG: list[tuple[str, int, list[int] | None, int, str]] = [
    ("SAVE10", 10, None, 0, "PCT"),
    ("SAVE15", 15, None, 2500, "PCT"),      # min $25
    ("SAVE20", 20, None, 5000, "PCT"),      # min $50
    ("SAVE25", 25, None, 7500, "PCT"),      # min $75
    ("CLEARANCE30", 30, None, 0, "PCT"),
    ("BOGO50", 50, None, 0, "BOGO"),        # buy one get one 50% off (qty >= 2)
    ("NEWYEAR15", 15, [1, 2], 0, "PCT"),
    ("SPRINGSALE20", 20, [3, 4, 5], 0, "PCT"),
    ("SUMMER25", 25, [6, 7, 8], 0, "PCT"),
    ("BACKTOSCHOOL20", 20, [8, 9], 0, "PCT"),
    ("BFRIDAY30", 30, [11], 0, "PCT"),
    ("BFRIDAY40", 40, [11], 10000, "PCT"),  # min $100
    ("HOLIDAY20", 20, [12], 0, "PCT"),
]
# Evergreen fallback: always eligible, no minimum, percentage only.
EVERGREEN: list[tuple[str, int]] = [
    ("SAVE10", 10), ("CLEARANCE30", 30), ("DEAL15", 15),
]

# Per-customer shopping segments (datagen CustomerJourney segment distribution).
SEGMENT_WEIGHTS: list[tuple[str, float]] = [
    ("BUDGET", 0.35), ("CONVENIENCE", 0.25), ("QUALITY", 0.20), ("BRAND_LOYAL", 0.20),
]


def _segment_price_skew(u: Column, seg: Column) -> Column:
    """Skew a uniform draw toward cheaper/pricier products by customer segment.

    BUDGET biases to the low (cheap) end of a department's price-ranked
    catalog, QUALITY/BRAND_LOYAL to the high end, CONVENIENCE is neutral.
    """
    return (F.when(seg == "BUDGET", u * u)
            .when(seg == "QUALITY", F.lit(1.0) - (F.lit(1.0) - u) * (F.lit(1.0) - u))
            .when(seg == "BRAND_LOYAL", F.pow(u, F.lit(0.85)))
            .otherwise(u))


# Weather simulation (datagen EventPatterns): a per-store-day weather state
# scales foot traffic. Seasonal odds make winter snow/storm and summer sun more
# likely. Only the traffic multiplier surfaces (there is no weather column).
WEATHER_MULTS = [1.1, 1.0, 0.7, 0.6, 0.5]  # sunny, cloudy, rainy, snowy, stormy
WEATHER_P_WINTER = [0.25, 0.30, 0.15, 0.20, 0.10]
WEATHER_P_SUMMER = [0.55, 0.25, 0.15, 0.00, 0.05]
WEATHER_P_SHOULDER = [0.40, 0.30, 0.20, 0.05, 0.05]

# Shopping trip archetypes (datagen ShoppingBehaviorType): each trip is a quick
# run / normal run / family shop / bulk stock-up, giving multi-modal basket
# sizes. Multipliers scale the store-type basket_lambda; weighted mean ~1.0 so
# overall item volume is preserved.
TRIP_TYPES: list[tuple[str, float, float]] = [  # (name, weight, basket multiplier)
    ("QUICK_TRIP", 0.30, 0.30),
    ("GROCERY_RUN", 0.40, 0.90),
    ("FAMILY_SHOPPING", 0.20, 1.60),
    ("BULK_SHOPPING", 0.10, 2.50),
]


def _cdf_pick_lit(u: Column, probs: list[float], values: list[float]) -> Column:
    """Inverse-CDF pick returning the chosen literal value."""
    acc = 0.0
    expr: Column | None = None
    for p, v in list(zip(probs, values))[:-1]:
        acc += p
        cond = u < F.lit(acc)
        expr = F.when(cond, F.lit(v)) if expr is None else expr.when(cond, F.lit(v))
    return expr.otherwise(F.lit(values[-1])) if expr is not None else F.lit(values[0])


def _weather_mult(u: Column, month: Column) -> Column:
    """Per-store-day weather traffic multiplier with seasonal weather odds."""
    return (F.when(month.isin(12, 1, 2),
                   _cdf_pick_lit(u, WEATHER_P_WINTER, WEATHER_MULTS))
            .when(month.isin(6, 7, 8),
                  _cdf_pick_lit(u, WEATHER_P_SUMMER, WEATHER_MULTS))
            .otherwise(_cdf_pick_lit(u, WEATHER_P_SHOULDER, WEATHER_MULTS)))


def _trip_basket_mult(u: Column) -> Column:
    """Pick a shopping-trip archetype's basket-size multiplier (inverse-CDF)."""
    return _cdf_pick_lit(u, [w for _, w, _ in TRIP_TYPES],
                         [m for _, _, m in TRIP_TYPES])


def _seasonal_factor(dept_name: str, month_col: Column) -> Column:
    """Monthly demand multiplier for a department (1.0 if no token matches)."""
    key = dept_name.lower()
    lifts = next((m for token, m in SEASONAL_LIFT.items() if token in key), None)
    if lifts is None:
        return F.lit(1.0)
    expr: Column | None = None
    for m in range(1, 13):
        cond = month_col == F.lit(m)
        expr = (F.when(cond, F.lit(lifts[m - 1])) if expr is None
                else expr.when(cond, F.lit(lifts[m - 1])))
    return expr.otherwise(F.lit(1.0))


def _with_seasonal_department(df: DataFrame, u_col: Column,
                              dept_weights: dict[str, float]) -> DataFrame:
    """Pick a department per row via inverse-CDF over base weights scaled by the
    seasonal lift for the row's month.

    Intermediate per-department weight (``_w*``) and cumulative-CDF (``_c*``)
    columns are materialized so Catalyst codegen stays small even for store
    types with many departments (an inline single expression overflows the
    64 KB JVM method limit and forces interpreted fallback). They are dropped
    before returning.
    """
    depts = list(dept_weights)
    month = F.month(F.col("event_date"))
    out = df
    for i, dn in enumerate(depts):
        out = out.withColumn(
            f"_w{i}", F.lit(float(dept_weights[dn])) * _seasonal_factor(dn, month))
    wt = F.col("_w0")
    for i in range(1, len(depts)):
        wt = wt + F.col(f"_w{i}")
    out = out.withColumn("_wt", wt)
    prev: str | None = None
    for i in range(len(depts) - 1):
        term = F.col(f"_w{i}") / F.col("_wt")
        out = out.withColumn(f"_c{i}", term if prev is None else F.col(prev) + term)
        prev = f"_c{i}"
    expr: Column | None = None
    for i, dn in enumerate(depts[:-1]):
        cond = u_col < F.col(f"_c{i}")
        expr = F.when(cond, F.lit(dn)) if expr is None else expr.when(cond, F.lit(dn))
    dept_col = expr.otherwise(F.lit(depts[-1])) if expr is not None else F.lit(depts[0])
    out = out.withColumn("department", dept_col)
    drop_cols = ([f"_w{i}" for i in range(len(depts))] + ["_wt"]
                 + [f"_c{i}" for i in range(len(depts) - 1)])
    return out.drop(*drop_cols)


def _promo_eligible_expr(idx_col: Column, month_col: Column,
                         ext_before: Column, qty: Column) -> Column:
    """True when the catalog entry at idx_col is eligible: in-month, line value
    meets any minimum, and BOGO needs qty >= 2."""
    expr: Column | None = None
    for i, (_, _, months, min_cents, kind) in enumerate(PROMO_CATALOG):
        e = F.lit(True) if months is None else month_col.isin(*months)
        if min_cents:
            e = e & (ext_before >= F.lit(min_cents))
        if kind == "BOGO":
            e = e & (qty >= F.lit(2))
        cond = idx_col == F.lit(i)
        expr = F.when(cond, e) if expr is None else expr.when(cond, e)
    return expr.otherwise(F.lit(False))


def _build_customer_activity_dates(spark: SparkSession, dims: dict[str, DataFrame],
                                   d: seeded_draws, cfg: GenerationConfig) -> DataFrame:
    """Generate customer activity end dates (churn dates) to create realistic gaps.
    
    ~25% of customers become "inactive" and have no purchases for 90+ days before
    the snapshot date. This creates a realistic churn cohort for ML training.
    """
    customers = dims["dim_customers"].select(F.col("ID").alias("customer_id"))
    
    # Generate a churn probability and inactive window for each customer
    # 25% churn rate; churned customers are inactive for 90-180 days before end_date
    churn_df = (
        customers
        .withColumn("_u_churn", d.u(["customer_id"], "churn_prob"))
        .withColumn("_is_churned", F.col("_u_churn") < F.lit(0.25))
        .withColumn("_days_inactive", F.when(
            F.col("_is_churned"),
            F.least(F.lit(180), F.greatest(F.lit(90),
                (d.h64(["customer_id"], "churn_window") % 91).cast("int") + 90))
        ).otherwise(F.lit(0)))
        .withColumn("customer_activity_end_date", F.when(
            F.col("_is_churned"),
            F.date_sub(F.to_date(F.lit(cfg.end_date)), F.col("_days_inactive"))
        ).otherwise(F.to_date(F.lit(cfg.end_date))))
        .select("customer_id", "customer_activity_end_date")
    )
    return churn_df


def _assign_customers(receipts: DataFrame, dims: dict[str, DataFrame],
                      d: seeded_draws, cfg: GenerationConfig) -> DataFrame:
    """Resolve each receipt's customer_id with store-geography affinity.

    With probability ``GEO_AFFINITY`` the customer is drawn from those living
    in the store's geography (resolved by a deterministic within-geography
    rank), otherwise a network-wide customer is used. Falls back to the
    network-wide pick when the store's geography has no customers.
    
    Only assigns customers to receipts if the receipt date is before the
    customer's activity end date (churn date), creating realistic customer
    lifecycle and churn patterns.
    """
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_id"), F.col("GeographyID").alias("cust_geo_id"))
    
    # Add customer activity end dates (churned customers have dates before snapshot)
    activity_dates = _build_customer_activity_dates(receipts.sparkSession, dims, d, cfg)
    customers = customers.join(activity_dates, "customer_id", "left")
    
    geo_sizes = customers.groupBy("cust_geo_id").agg(F.count("*").alias("geo_cust_count"))
    rank_w = Window.partitionBy("cust_geo_id").orderBy("customer_id")
    cust_ranked = customers.withColumn("local_rank", F.row_number().over(rank_w))

    r = (
        receipts
        .join(geo_sizes, receipts["store_geo_id"] == geo_sizes["cust_geo_id"], "left")
        .drop("cust_geo_id")
        .withColumn("geo_cust_count", F.coalesce(F.col("geo_cust_count"), F.lit(0)))
        .withColumn("global_customer", (d.h64(["receipt_id_ext"], "cust")
                                        % F.lit(cfg.customer_count) + 1).cast("long"))
        .withColumn("want_local",
                    (d.u(["receipt_id_ext"], "affinity") < F.lit(GEO_AFFINITY))
                    & (F.col("geo_cust_count") > 0))
        .withColumn("local_rank_target", F.when(
            F.col("geo_cust_count") > 0,
            (d.h64(["receipt_id_ext"], "localcust") % F.col("geo_cust_count") + 1))
            .cast("long"))
    )
    local = cust_ranked.select(
        F.col("cust_geo_id").alias("_lgeo"), F.col("local_rank").alias("_lrank"),
        F.col("customer_id").alias("local_customer"), 
        F.col("customer_activity_end_date").alias("_activity_end_date"))
    return (
        r.join(local, (F.col("store_geo_id") == F.col("_lgeo"))
               & (F.col("local_rank_target") == F.col("_lrank")), "left")
        .withColumn("customer_id", F.when(
            F.col("want_local") & F.col("local_customer").isNotNull(),
            F.col("local_customer")).otherwise(F.col("global_customer")))
        # Filter: only use customer if receipt date is before their activity end date
        .filter(F.col("event_date") <= F.coalesce(F.col("_activity_end_date"), F.to_date(F.lit(cfg.end_date))))
        .drop("_lgeo", "_lrank", "local_customer", "global_customer",
              "want_local", "local_rank_target", "geo_cust_count", "_activity_end_date")
    )


def _fmt(cents_col: Column) -> Column:
    """Integer cents -> 'XX.XX' string (no thousands separators)."""
    return F.format_string("%.2f", cents_col / F.lit(100.0))


def _pick_hour(u: Column, hourly_weights: list[float]) -> Column:
    """Inverse-CDF hour pick over the 24 relative weights."""
    total = sum(hourly_weights)
    chain: Column | None = None
    acc = 0.0
    for h, w in enumerate(hourly_weights[:-1]):
        acc += w / total
        cond = u < F.lit(acc)
        chain = F.when(cond, F.lit(h)) if chain is None else chain.when(cond, F.lit(h))
    return F.lit(23) if chain is None else chain.otherwise(F.lit(23))


def generate_receipts_group(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_receipts, fact_receipt_lines, fact_payments (in-store only)."""

    d = seeded_draws(cfg.seed)

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"), "tax_rate", "daily_traffic_multiplier",
        F.col("GeographyID").alias("store_geo_id"))
    store_ids = [r.store_id for r in stores.select("store_id").collect()]
    grid = store_day_grid(
        spark, store_ids, cfg.start_date, cfg.end_date, cfg.seed, "receipts"
    ).join(stores, "store_id")

    # --- per store-day receipt counts: base * daily * monthly * store multiplier
    dw = profile.daily_weights
    mw = profile.monthly_weights
    d_mean = sum(dw) / 7.0
    m_mean = sum(mw) / 12.0
    # dayofweek: 1=Sunday..7=Saturday; profile lists Monday-first -> Mon=1..Sun=7
    daily_w = F.element_at(
        F.array(*[F.lit(w / d_mean) for w in dw]),
        ((F.dayofweek(F.col("day")) + 5) % 7) + 1,
    )
    monthly_w = F.element_at(F.array(*[F.lit(w / m_mean) for w in mw]), F.month("day"))
    lam = (F.lit(float(cfg.transactions_per_store_day)) * daily_w * monthly_w
           * F.col("daily_traffic_multiplier")
           * _weather_mult(d.u(["store_id", "day"], "weather"), F.month("day")))
    n_rcpt = F.greatest(
        F.lit(1), F.round(lam + d.gauss(["store_id", "day"], "n") * F.sqrt(lam)))
    grid = grid.withColumn("n_receipts", n_rcpt.cast("int"))

    # --- explode to receipts; hour from hourly weights (inverse CDF over 24 bins)
    receipts = (
        grid.withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_receipts"))))
        .withColumn("hour", _pick_hour(
            d.u(["store_id", "day", "seq"], "hour"), profile.hourly_weights))
        .withColumn("minute", (d.h64(["store_id", "day", "seq"], "min") % 60).cast("int"))
        .withColumn("second", (d.h64(["store_id", "day", "seq"], "sec") % 60).cast("int"))
        .withColumn("event_ts", F.make_timestamp(
            F.year("day"), F.month("day"), F.dayofmonth("day"),
            F.col("hour"), F.col("minute"), F.col("second")))
        .withColumn("event_date", F.col("day"))
        # RCP(3) + yyyyMMddHHmm(12) + store(4) + seq(6) = 25 chars, unique by
        # construction (store_id, day, seq) is a key
        .withColumn("receipt_id_ext", F.concat(
            F.lit("RCP"), F.date_format("event_ts", "yyyyMMddHHmm"),
            F.lpad(F.col("store_id").cast("string"), 4, "0"),
            F.lpad(F.col("seq").cast("string"), 6, "0")))
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
    )

    # geography affinity: resolve each receipt's customer (local vs network-wide)
    receipts = _assign_customers(receipts, dims, d, cfg)

    # per-customer shopping segment (datagen CustomerJourney): drives basket size
    # and price-tier preference, so a customer behaves consistently across trips.
    seg_basket = (F.when(F.col("_seg") == "CONVENIENCE", 0.7)
                  .when(F.col("_seg") == "QUALITY", 1.15)
                  .when(F.col("_seg") == "BRAND_LOYAL", 1.4)
                  .otherwise(1.0))
    # trip archetype gives multi-modal basket sizes (quick vs bulk stock-up)
    lam_b = (F.lit(float(profile.basket_lambda)) * seg_basket
             * _trip_basket_mult(d.u(["receipt_id_ext"], "trip")))
    receipts = (
        receipts
        .withColumn("_seg", d.pick_by_weights(["customer_id"], "seg", SEGMENT_WEIGHTS))
        .withColumn("basket_n", F.greatest(F.lit(1), F.round(
            lam_b + d.gauss(["receipt_id_ext"], "basket") * F.sqrt(lam_b))).cast("int"))
        .withColumn("tender_type", d.pick_by_weights(
            ["receipt_id_ext"], "tender", [(n, w) for n, w, _, _, _ in TENDERS]))
    )

    # --- lines: explode baskets, weighted department -> uniform product within dept
    products = dims["dim_products"].select(
        F.col("ID").alias("product_id"), F.col("SalePrice"), F.col("taxability"),
        F.col("Department").alias("department"))

    product_departments = {r.department for r in products.select("department").distinct().collect()}
    unknown = set(profile.department_weights) - product_departments
    if unknown:
        raise ValueError(
            f"profile department_weights reference departments missing from the catalog: {sorted(unknown)}"
        )

    # rank products by price within department so the segment skew can target a
    # cheap/pricey tier (datagen CustomerJourney price preference)
    pw = Window.partitionBy("department").orderBy("SalePrice", "product_id")
    products_ranked = products.withColumn("dept_rank", F.row_number().over(pw))
    dept_sizes = products.groupBy("department").agg(F.count("*").alias("dept_size"))

    exploded = (
        receipts.select("receipt_id_ext", "event_ts", "event_date", "store_id",
                        "tax_rate", "basket_n", "_seg")
        .withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
    )
    exploded = _with_seasonal_department(
        exploded, d.u(["receipt_id_ext", "line_num"], "dept"),
        profile.department_weights)

    lines = (
        exploded
        .join(dept_sizes, "department")
        .withColumn("_pskew", _segment_price_skew(
            d.u(["receipt_id_ext", "line_num"], "prod"), F.col("_seg")))
        .withColumn("dept_rank", F.least(F.col("dept_size"), F.greatest(F.lit(1),
            (F.floor(F.col("_pskew") * F.col("dept_size")) + 1).cast("int"))))
        .join(products_ranked, ["department", "dept_rank"])
        .withColumn("quantity", F.greatest(F.lit(1), F.least(F.lit(5), F.round(
            d.u(["receipt_id_ext", "line_num"], "qty") * 3 + 0.7).cast("int"))))
        .withColumn("unit_cents", F.round(F.col("SalePrice") * 100).cast("long"))
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn("has_promo", d.u(["receipt_id_ext", "line_num"], "promo")
                    < F.lit(profile.promo_rate))
        # named promo code + matching discount (datagen promotion_utils parity):
        # a seasonal/min-purchase/BOGO code if eligible, else an evergreen code.
        .withColumn("_pidx", (d.h64(["receipt_id_ext", "line_num"], "pcode")
                              % F.lit(len(PROMO_CATALOG))).cast("int"))
        .withColumn("_pcode", F.element_at(
            F.array(*[F.lit(t[0]) for t in PROMO_CATALOG]), F.col("_pidx") + 1))
        .withColumn("_ppct", F.element_at(
            F.array(*[F.lit(t[1]) for t in PROMO_CATALOG]), F.col("_pidx") + 1))
        .withColumn("_pkind", F.element_at(
            F.array(*[F.lit(t[4]) for t in PROMO_CATALOG]), F.col("_pidx") + 1))
        .withColumn("_pelig", _promo_eligible_expr(
            F.col("_pidx"), F.month(F.col("event_date")),
            F.col("ext_before"), F.col("quantity")))
        .withColumn("_evidx", (d.h64(["receipt_id_ext", "line_num"], "pcodeev")
                               % F.lit(len(EVERGREEN))).cast("int"))
        .withColumn("_evcode", F.element_at(
            F.array(*[F.lit(c) for c, _ in EVERGREEN]), F.col("_evidx") + 1))
        .withColumn("_evpct", F.element_at(
            F.array(*[F.lit(p) for _, p in EVERGREEN]), F.col("_evidx") + 1))
        .withColumn("promo_code", F.when(F.col("has_promo"), F.when(
            F.col("_pelig"), F.col("_pcode")).otherwise(F.col("_evcode"))))
        .withColumn("_disc_pct", F.when(
            F.col("_pelig"), F.col("_ppct")).otherwise(F.col("_evpct")))
        .withColumn("_disc_kind", F.when(
            F.col("_pelig"), F.col("_pkind")).otherwise(F.lit("PCT")))
        .withColumn("discount_cents", F.when(F.col("has_promo"),
            F.when(F.col("_disc_kind") == "BOGO",
                   # buy-one-get-one: every 2nd item discounted at _disc_pct
                   F.floor(F.floor(F.col("quantity") / F.lit(2))
                           * F.col("unit_cents") * F.col("_disc_pct")
                           / F.lit(100.0) + F.lit(0.5)).cast("long"))
            .otherwise(F.floor(
                F.col("ext_before") * F.col("_disc_pct") / F.lit(100.0)
                + F.lit(0.5)).cast("long")))
            .otherwise(F.lit(0).cast("long")))
        .withColumn("ext_cents",
                    F.greatest(F.lit(0).cast("long"),
                               F.col("ext_before") - F.col("discount_cents")))
        # tax: integer basis-point math, replicating datagen _tax_cents exactly
        .withColumn("rate_bps", F.round(F.col("tax_rate") * 10000).cast("long"))
        .withColumn("tax_mult", F.when(F.col("taxability") == "TAXABLE", 100)
                    .when(F.col("taxability") == "REDUCED_RATE", 50)
                    .otherwise(0).cast("long"))
        .withColumn("line_tax_cents", F.floor(
            (F.col("ext_cents") * F.col("rate_bps") * F.col("tax_mult")
             + F.lit(500_000)) / F.lit(1_000_000)).cast("long"))
    )

    fact_receipt_lines = lines.select(
        "receipt_id_ext", "event_ts", "event_date", "line_num", "product_id",
        "quantity", _fmt(F.col("unit_cents")).alias("unit_price"), "unit_cents",
        _fmt(F.col("ext_cents")).alias("ext_price"), "ext_cents", "promo_code",
    ).select(*column_names("fact_receipt_lines"))

    # --- header rollup
    hdr = lines.groupBy("receipt_id_ext").agg(
        F.sum("ext_cents").alias("subtotal_cents"),
        F.sum("discount_cents").alias("discount_cents"),
        F.sum("line_tax_cents").alias("tax_cents"))
    fact_receipts = (
        receipts.join(hdr, "receipt_id_ext")
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("receipt_type", F.lit("SALE"))
        .withColumn("payment_method", F.col("tender_type"))
        .select(
            "receipt_id_ext", "trace_id", "event_ts", "event_date", "store_id",
            "customer_id", "receipt_type", "tender_type", "subtotal_cents",
            _fmt(F.col("discount_cents")).alias("discount_amount"), "tax_cents",
            "total_cents", _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
            _fmt(F.col("tax_cents")).alias("tax_amount"),
            _fmt(F.col("total_cents")).alias("total_amount"), "payment_method",
            # Legacy semantic-model column (sourceColumn: Subtotal); same value
            # as subtotal_amount per the TMDL contract.
            _fmt(F.col("subtotal_cents")).alias("Subtotal"),
        ).select(*column_names("fact_receipts"))
    )

    # --- payments (one per receipt; in-store, so order_id_ext is NULL)
    u_dec = d.u(["receipt_id_ext"], "decline")
    decline_p: Column = F.lit(0.0)
    for name, _, mult, _, _ in TENDERS:
        decline_p = F.when(
            F.col("payment_method") == name, F.lit(BASE_DECLINE * mult)
        ).otherwise(decline_p)
    proc_lo: Column = F.lit(TENDERS[-1][3])
    proc_hi: Column = F.lit(TENDERS[-1][4])
    for name, _, _, lo, hi in TENDERS[:-1]:
        proc_lo = F.when(F.col("payment_method") == name, F.lit(lo)).otherwise(proc_lo)
        proc_hi = F.when(F.col("payment_method") == name, F.lit(hi)).otherwise(proc_hi)
    reason_idx = (d.h64(["receipt_id_ext"], "reason") % len(DECLINE_REASONS)).cast("int")
    fact_payments = (
        fact_receipts
        .withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("transaction_id", F.concat(
            F.lit("TXN_"), F.unix_timestamp("event_ts").cast("string"), F.lit("_"),
            F.lpad((d.h64(["receipt_id_ext"], "txn") % 1_000_000).cast("string"), 6, "0")))
        .withColumn("status",
                    F.when(u_dec < decline_p, "DECLINED").otherwise("APPROVED"))
        .withColumn("decline_reason", F.when(
            F.col("status") == "DECLINED",
            F.element_at(F.array(*[F.lit(r) for r in DECLINE_REASONS]),
                         reason_idx + 1)))
        .withColumn("processing_time_ms",
                    (proc_lo + d.u(["receipt_id_ext"], "proc")
                     * (proc_hi - proc_lo)).cast("long"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/returns.py ---
"""Returns fact group, Spark-native (unioned into the receipts contract).

Semantics (datagen utils_mixin): sample ~``cfg.return_rate`` of SALE receipts
per day — Dec 26 spikes 6x, capped at 10% of the day's receipts. The return
header gets a new ``receipt_id_ext`` with the same 25-char layout as sales
(``RET`` + yyyyMMddHHmm + store4 + seq6), ``receipt_type='RETURN'``, noon
``event_ts`` on the same day, NULL ``customer_id``, CREDIT_CARD tender, and
negated cents. Lines mirror the original receipt's lines with negative
quantity / ext_cents (``promo_code='RETURN'``); each return gets one negative,
APPROVED payment — refunds don't decline.

All randomness comes from `runtime.seeded_draws` keyed on the original
receipt id, so output is deterministic for a (config, seed) pair.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# CREDIT_CARD processing-time bounds (ms), matching the sales TENDERS table.
_PROC_MS_LO = 1500
_PROC_MS_HI = 4000


def generate_returns(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate RETURN fact_receipts / fact_receipt_lines / fact_payments."""

    d = seeded_draws(cfg.seed)

    sales = sales_group["fact_receipts"].filter(F.col("receipt_type") == "SALE")

    # --- sample: per-day rate, 6x on Dec 26, hard-capped at 10%.
    # Deterministic rank-based sampling (lowest seeded u first, exactly
    # floor(day_rate * n_day) per day) rather than per-row Bernoulli
    # `u < day_rate`: with Bernoulli the Dec-26 expectation equals the cap,
    # which equals exactly 2x the nominal rate, so the spike contract is a
    # coin flip on hash noise. Ranking pins other days at/below nominal and
    # lands Dec 26 on the cap, keeping output deterministic per (config, seed).
    day_rate = F.least(
        F.lit(0.10),
        F.lit(cfg.return_rate)
        * F.when(
            (F.month("event_date") == 12) & (F.dayofmonth("event_date") == 26),
            F.lit(6.0),
        ).otherwise(F.lit(1.0)),
    )
    u_ret = d.u(["receipt_id_ext"], "return")
    day_w = Window.partitionBy("event_date")
    sampled = (
        sales
        .withColumn("_n_keep", F.floor(day_rate * F.count("*").over(day_w)))
        .withColumn("_ret_rank", F.row_number().over(
            day_w.orderBy(u_ret, "receipt_id_ext")))
        .filter(F.col("_ret_rank") <= F.col("_n_keep"))
        .drop("_n_keep", "_ret_rank")
    )

    # --- header: new RET id (RET(3)+yyyyMMddHHmm(12)+store4+seq6 = 25 chars,
    # unique because seq is a row_number per (store_id, event_date) at noon)
    seq_w = Window.partitionBy("store_id", "event_date").orderBy("orig_receipt_id_ext")
    header = (
        sampled
        .withColumnRenamed("receipt_id_ext", "orig_receipt_id_ext")
        .withColumn("event_ts", F.to_timestamp(
            F.concat(F.col("event_date").cast("string"), F.lit(" 12:00:00"))))
        .withColumn("seq", F.row_number().over(seq_w))
        .withColumn("receipt_id_ext", F.concat(
            F.lit("RET"), F.date_format("event_ts", "yyyyMMddHHmm"),
            F.lpad(F.col("store_id").cast("string"), 4, "0"),
            F.lpad(F.col("seq").cast("string"), 6, "0")))
        .withColumn("trace_id", F.concat(F.lit("TRC"), F.col("receipt_id_ext")))
        .withColumn("customer_id", F.lit(None).cast("long"))
        .withColumn("receipt_type", F.lit("RETURN"))
        .withColumn("tender_type", F.lit("CREDIT_CARD"))
        .withColumn("payment_method", F.lit("CREDIT_CARD"))
        .withColumn("subtotal_cents", -F.col("subtotal_cents"))
        .withColumn("tax_cents", -F.col("tax_cents"))
        .withColumn("total_cents", -F.col("total_cents"))
    )

    fact_receipts = header.select(
        "receipt_id_ext", "trace_id", "event_ts", "event_date", "store_id",
        "customer_id", "receipt_type", "tender_type", "subtotal_cents",
        F.lit("0.00").alias("discount_amount"), "tax_cents", "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"), "payment_method",
        # Legacy semantic-model column mirrors subtotal_amount (TMDL contract).
        _fmt(F.col("subtotal_cents")).alias("Subtotal"),
    ).select(*column_names("fact_receipts"))

    # --- lines: mirror the original receipt's lines, negated
    orig_lines = sales_group["fact_receipt_lines"].select(
        F.col("receipt_id_ext").alias("orig_receipt_id_ext"),
        "line_num", "product_id", "quantity", "unit_cents", "ext_cents",
    )
    fact_receipt_lines = (
        header.select("orig_receipt_id_ext", "receipt_id_ext",
                      "event_ts", "event_date")
        .join(orig_lines, "orig_receipt_id_ext")
        .withColumn("quantity", (-F.col("quantity")).cast("int"))
        .withColumn("ext_cents", -F.col("ext_cents"))
        .withColumn("promo_code", F.lit("RETURN"))
        .select(
            "receipt_id_ext", "event_ts", "event_date", "line_num", "product_id",
            "quantity", _fmt(F.col("unit_cents")).alias("unit_price"), "unit_cents",
            _fmt(F.col("ext_cents")).alias("ext_price"), "ext_cents", "promo_code",
        ).select(*column_names("fact_receipt_lines"))
    )

    # --- payments: one negative APPROVED CREDIT_CARD refund per return
    fact_payments = (
        fact_receipts
        .withColumn("order_id_ext", F.lit(None).cast("string"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .withColumn("transaction_id", F.concat(
            F.lit("TXN_"), F.unix_timestamp("event_ts").cast("string"), F.lit("_"),
            F.lpad((d.h64(["receipt_id_ext"], "txn") % 1_000_000).cast("string"),
                   6, "0")))
        .withColumn("status", F.lit("APPROVED"))
        .withColumn("decline_reason", F.lit(None).cast("string"))
        .withColumn("processing_time_ms", (
            F.lit(_PROC_MS_LO) + d.u(["receipt_id_ext"], "proc")
            * F.lit(_PROC_MS_HI - _PROC_MS_LO)).cast("long"))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_receipts": fact_receipts,
        "fact_receipt_lines": fact_receipt_lines,
        "fact_payments": fact_payments,
    }

# --- retail_setup/generation/store_activity.py ---
"""Store ops + foot traffic facts, Spark-native.

fact_store_ops: exactly two rows (opened/closed) per store-day, with the
open/close hours parsed from `dim_stores.operating_hours` over the four
known formats ("6-22", "7-22", "7-23", "24h"). Dec 25 is skipped entirely
(stores closed for Christmas, any year).

fact_foot_traffic is derived from receipts: per store-hour receipt counts
are inflated by an inverse conversion rate (peak-hour and weekend adjusted)
into total visitors, then split across the five sensor zones with a
store_format share table. Dwell times are uniform draws within per-zone
ranges via `runtime.seeded_draws`, so output is deterministic for a
(config, seed) pair regardless of cluster shape.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# operating_hours literal -> (open_hour, close_hour); "24h" maps to 0/24
OPERATING_HOURS_MAP = {"6-22": (6, 22), "7-22": (7, 22), "7-23": (7, 23), "24h": (0, 24)}

BASE_CONVERSION = 0.20
PEAK_HOURS = (12, 13, 17, 18, 19)
PEAK_MULTIPLIER = 1.3
WEEKEND_MULTIPLIER = 0.9
# Baseline browsers per store-hour (scaled by store size + hour-of-day) so that
# open hours with zero receipts still emit foot traffic.
BASE_HOURLY_BROWSERS = 8

# zone -> (dwell_lo_seconds, dwell_hi_seconds)
ZONE_DWELL = {
    "ENTRANCE_MAIN": (45, 90),
    "ENTRANCE_SIDE": (30, 75),
    "AISLES_A": (180, 420),
    "AISLES_B": (120, 300),
    "CHECKOUT": (90, 240),
}
ZONES = list(ZONE_DWELL)

# store_format -> zone share, ordered as ZONES
FORMAT_ZONE_SHARES = {
    "hypermarket": [0.20, 0.10, 0.35, 0.25, 0.10],
    "superstore": [0.25, 0.10, 0.30, 0.25, 0.10],
    "standard": [0.30, 0.15, 0.25, 0.15, 0.15],
    "neighborhood": [0.35, 0.15, 0.20, 0.10, 0.20],
    "express": [0.45, 0.15, 0.10, 0.05, 0.25],
}


def generate_store_ops(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Two rows (opened/closed) per store-day, skipping Dec 25 of any year."""

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"), "operating_hours")
    store_ids = [r.store_id for r in stores.select("store_id").collect()]

    grid = (
        store_day_grid(spark, store_ids, cfg.start_date, cfg.end_date,
                       cfg.seed, "store_ops")
        .filter(~((F.month("day") == 12) & (F.dayofmonth("day") == 25)))
        .join(stores, "store_id")
    )

    # operating_hours -> open/close hour via a F.when chain over known formats
    open_hour, close_hour = None, None
    for literal, (o, c) in OPERATING_HOURS_MAP.items():
        cond = F.col("operating_hours") == literal
        open_hour = F.when(cond, o) if open_hour is None else open_hour.when(cond, o)
        close_hour = F.when(cond, c) if close_hour is None else close_hour.when(cond, c)
    # unknown formats fall back to standard hours instead of silent NULL event_ts
    grid = grid.withColumn("open_hour", open_hour.otherwise(6)).withColumn(
        "close_hour", close_hour.otherwise(22))

    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    open_ts = F.timestamp_seconds(day_ts + F.col("open_hour") * 3600)
    # 24h stores "close" at 23:59:00 the same day; others at close_hour:00
    close_secs = F.when(
        F.col("close_hour") == 24, F.lit(23 * 3600 + 59 * 60)
    ).otherwise(F.col("close_hour") * 3600)
    close_ts = F.timestamp_seconds(day_ts + close_secs)

    ops = grid.select(
        "store_id",
        F.col("day").alias("event_date"),
        F.explode(F.array(
            F.struct(F.lit("opened").alias("operation_type"), open_ts.alias("event_ts")),
            F.struct(F.lit("closed").alias("operation_type"), close_ts.alias("event_ts")),
        )).alias("op"),
    ).select(
        F.col("op.event_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-OPS-"), F.col("store_id"), F.lit("-"),
            F.col("event_date"), F.lit("-"), F.col("op.operation_type"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.col("op.operation_type").alias("operation_type"),
        "event_date",
    )
    return ops.select(*column_names("fact_store_ops"))


def generate_foot_traffic(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Per store-hour-zone sensor counts.

    Foot traffic is generated for every open store-hour (an independent grid
    derived from ``operating_hours``), not only hours that have receipts, so
    zero-receipt browsing traffic exists and ``foot_traffic >= receipts`` holds
    per store-hour. Receipt-bearing hours keep the receipt-derived visitor
    count; zero-receipt open hours get a size/hour-scaled browsing baseline.
    """

    d = seeded_draws(cfg.seed)

    stores = dims["dim_stores"].select(
        F.col("ID").alias("store_id"), "store_format", "operating_hours",
        "daily_traffic_multiplier")
    store_ids = [r.store_id for r in stores.select("store_id").collect()]

    # operating_hours -> open/close hour via a F.when chain over known formats
    open_hour, close_hour = None, None
    for literal, (o, c) in OPERATING_HOURS_MAP.items():
        cond = F.col("operating_hours") == literal
        open_hour = F.when(cond, o) if open_hour is None else open_hour.when(cond, o)
        close_hour = F.when(cond, c) if close_hour is None else close_hour.when(cond, c)

    # independent store-open-hour grid (skip Dec 25 — stores closed, see store_ops)
    grid = (
        store_day_grid(spark, store_ids, cfg.start_date, cfg.end_date,
                       cfg.seed, "foot_traffic")
        .filter(~((F.month("day") == 12) & (F.dayofmonth("day") == 25)))
        .join(stores, "store_id")
        .withColumn("open_hour", open_hour.otherwise(6))
        .withColumn("close_hour", close_hour.otherwise(22))
        .withColumn("hour", F.explode(
            F.sequence(F.col("open_hour"), F.col("close_hour") - 1)))
        .withColumn("hour_ts", F.timestamp_seconds(
            F.unix_timestamp(F.col("day").cast("timestamp")) + F.col("hour") * 3600))
    )

    hourly_receipts = receipts.groupBy(
        "store_id", F.date_trunc("hour", "event_ts").alias("hour_ts")
    ).agg(F.count("*").alias("receipts"))

    # full outer = union of open hours (browsing) and receipt hours (coverage);
    # store attributes are re-joined on store_id so they are never null.
    store_attrs = dims["dim_stores"].select(
        F.col("ID").alias("store_id"), "store_format", "daily_traffic_multiplier")
    hourly = (
        grid.select("store_id", "hour_ts")
        .join(hourly_receipts, ["store_id", "hour_ts"], "full_outer")
        .withColumn("receipts", F.coalesce(F.col("receipts"), F.lit(0)))
        .join(store_attrs, "store_id")
    )

    hour = F.hour("hour_ts")
    weekend = F.dayofweek("hour_ts").isin(1, 7)  # Sunday=1, Saturday=7
    conv = (
        F.lit(BASE_CONVERSION)
        * F.when(hour.isin(*PEAK_HOURS), PEAK_MULTIPLIER).otherwise(1.0)
        * F.when(weekend, WEEKEND_MULTIPLIER).otherwise(1.0)
    )
    receipt_derived = F.greatest(
        (F.col("receipts") + 1).cast("long"),
        F.round(F.col("receipts") / conv).cast("long"),
    )
    # zero-receipt hours: size/hour-scaled browsing baseline (>= 1)
    hour_weight = (F.when(hour.isin(*PEAK_HOURS), 1.5)
                   .when((hour < 9) | (hour >= 21), 0.5).otherwise(1.0))
    baseline = F.greatest(F.lit(1), F.round(
        F.col("daily_traffic_multiplier") * hour_weight * F.lit(BASE_HOURLY_BROWSERS)))
    total = F.when(F.col("receipts") > 0, receipt_derived).otherwise(baseline)
    hourly = hourly.withColumn("total_visitors", total.cast("long"))

    # per-format share column for each zone, then explode the 5-zone structs
    zone_structs = []
    for i, zone in enumerate(ZONES):
        share = None
        for fmt, shares in FORMAT_ZONE_SHARES.items():
            cond = F.col("store_format") == fmt
            share = (F.when(cond, shares[i]) if share is None
                     else share.when(cond, shares[i]))
        lo, hi = ZONE_DWELL[zone]
        zone_structs.append(F.struct(
            F.lit(zone).alias("zone"),
            share.alias("share"),
            F.lit(lo).alias("dwell_lo"),
            F.lit(hi).alias("dwell_hi"),
        ))

    ft = hourly.select(
        "store_id", "hour_ts", "total_visitors",
        F.explode(F.array(*zone_structs)).alias("z"),
    )

    dwell_u = d.u([F.col("store_id"), F.col("hour_ts"), F.col("z.zone")], "ft_dwell")
    ft = ft.select(
        F.col("hour_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-FT-"), F.col("store_id"), F.lit("-"),
            F.date_format("hour_ts", "yyyyMMddHH"), F.lit("-"), F.col("z.zone"),
        ).alias("trace_id"),
        F.col("store_id").cast("long").alias("store_id"),
        F.format_string("SENSOR_%03d_%s", F.col("store_id"), F.col("z.zone"))
            .alias("sensor_id"),
        F.col("z.zone").alias("zone"),
        F.floor(
            F.col("z.dwell_lo") + dwell_u * (F.col("z.dwell_hi") - F.col("z.dwell_lo") + 1)
        ).cast("long").alias("dwell_seconds"),
        F.round(F.col("total_visitors") * F.col("z.share")).cast("long").alias("count"),
        F.to_date("hour_ts").alias("event_date"),
    )
    return ft.select(*column_names("fact_foot_traffic"))

# --- retail_setup/generation/online_orders.py ---
"""Online orders fact group (headers, lines, payments stream), Spark-native.

Mirrors `receipts.py`: all randomness flows through `runtime.seeded_draws`
(xxhash64-based, partition-arrangement independent), money is integer cents,
and daily volume is a clamped-normal approximation of Poisson.

Documented decisions (per plan 2b, Task 5):
- Tax uses the exact integer basis-point formula from receipts. Store/BOPIS
  lines use the fulfilment store's rate; DC-shipped lines use the customer's
  geography rate, with the network mean as fallback.
- Backorders: a small fraction of non-BOPIS lines ship but are not yet
  delivered (fulfillment_status SHIPPED, delivered_ts NULL), mirroring
  datagen's ``"DELIVERED" if not is_backordered else "SHIPPED"``.
- Product pick is uniform over the full catalog — online ignores the profile's
  department weights.
- CANCELLED orders get NO payment row (no pay-then-refund), which preserves
  the `payment.amount == header.total` invariant.
"""

from datetime import timedelta

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Online tender mix per plan: 60% CC / 25% DC / 10% PAYPAL / 5% OTHER.
# (method, mix weight, decline multiplier, processing_ms lo, processing_ms hi)
# CC/DC constants carried from the Plan-2a TENDERS table; PAYPAL/OTHER are the
# online-only additions (PAYPAL x1.1 decline, OTHER x0.5 decline).
ONLINE_TENDERS = [
    ("CREDIT_CARD", 0.60, 1.0, 1500, 4000),
    ("DEBIT_CARD", 0.25, 0.8, 1200, 3500),
    ("PAYPAL", 0.10, 1.1, 2000, 5000),
    ("OTHER", 0.05, 0.5, 1500, 4000),
]

CANCEL_RATE = 0.02
# Fraction of shippable (non-BOPIS) lines that backorder: ship but not yet
# delivered. Approximates datagen's inventory-driven backorder check, which
# utility cannot evaluate here (the inventory chain is generated afterwards).
BACKORDER_RATE = 0.08

# basket-size buckets: 60% -> 1-3 lines, 30% -> 2-5, 10% -> 5-8
BASKET_BUCKETS = [("S", 0.60, 1, 3), ("M", 0.30, 2, 5), ("L", 0.10, 5, 8)]

# promo codes with matching percentage discounts (5/10/20%)
PROMOS = [("PROMO05", 5), ("PROMO10", 10), ("PROMO20", 20)]


def generate_online_orders(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    profile: StoreTypeProfile,
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate fact_online_order_headers, fact_online_order_lines, payments.

    Payments are returned under the "payments" key for the orchestrator to
    union with the in-store stream before writing fact_payments.
    """
    d = seeded_draws(cfg.seed)

    # Destination-based tax: per-customer-geography rate (mean store tax_rate in
    # that geography), with the network mean as the fallback for geographies
    # that have no store.
    mean_rate = dims["dim_stores"].agg(F.avg("tax_rate")).first()[0]
    mean_bps = int(round(float(mean_rate) * 10000))
    geo_rate = dims["dim_stores"].groupBy("GeographyID").agg(
        F.avg("tax_rate").alias("geo_rate"))
    cust_rate = (
        dims["dim_customers"].select(F.col("ID").alias("customer_id"), "GeographyID")
        .join(geo_rate, "GeographyID", "left")
        .select(
            "customer_id",
            F.coalesce(F.round(F.col("geo_rate") * 10000), F.lit(mean_bps))
            .cast("long").alias("rate_bps"),
        )
    )

    dc_ids = sorted(r.ID for r in dims["dim_distribution_centers"].select("ID").collect())
    store_ids = sorted(r.ID for r in dims["dim_stores"].select("ID").collect())
    dc_arr = F.array(*[F.lit(int(i)).cast("long") for i in dc_ids])
    st_arr = F.array(*[F.lit(int(i)).cast("long") for i in store_ids])

    # --- day grid: network-wide daily volume, monthly-weight scaled, clamped normal
    mw = profile.monthly_weights
    m_mean = sum(mw) / 12.0
    day_list = [
        cfg.start_date + timedelta(days=i)
        for i in range((cfg.end_date - cfg.start_date).days + 1)
    ]
    days = spark.createDataFrame([(day,) for day in day_list], "day date")
    monthly_w = F.element_at(F.array(*[F.lit(w / m_mean) for w in mw]), F.month("day"))
    lam = F.lit(float(cfg.online_orders_per_day)) * monthly_w
    n_orders = F.greatest(
        F.lit(1), F.round(lam + d.gauss(["day"], "onl_n") * F.sqrt(lam))).cast("int")

    # --- orders: ids, timestamps, customer, tender, cancellation, basket size
    bucket = d.pick_by_weights(
        ["day", "seq"], "onl_bucket", [(name, w) for name, w, _, _ in BASKET_BUCKETS])
    basket_n: Column = F.lit(None).cast("int")
    for name, _, lo, hi in BASKET_BUCKETS:
        basket_n = F.when(
            bucket == name,
            (d.h64(["day", "seq"], f"onl_size_{name}") % (hi - lo + 1) + lo).cast("int"),
        ).otherwise(basket_n)

    orders = (
        days.withColumn("n_orders", n_orders)
        .withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_orders"))))
        .withColumn("hour", (d.h64(["day", "seq"], "onl_hour") % 24).cast("int"))
        .withColumn("minute", (d.h64(["day", "seq"], "onl_min") % 60).cast("int"))
        .withColumn("second", (d.h64(["day", "seq"], "onl_sec") % 60).cast("int"))
        .withColumn("event_ts", F.make_timestamp(
            F.year("day"), F.month("day"), F.dayofmonth("day"),
            F.col("hour"), F.col("minute"), F.col("second")))
        .withColumn("event_date", F.col("day"))
        # ONL + yyyyMMdd + 5-digit seq + 3-digit draw; unique because (day, seq)
        # is a key within the grid.
        # NOTE: lpad(seq, 5) collides above 99,999 orders/day; config max ~16k so safe.
        .withColumn("order_id_ext", F.concat(
            F.lit("ONL"), F.date_format("day", "yyyyMMdd"),
            F.lpad(F.col("seq").cast("string"), 5, "0"),
            F.lpad((d.h64(["day", "seq"], "onl_rand") % 1000).cast("string"), 3, "0")))
        .withColumn("customer_id", (d.h64(["order_id_ext"], "onl_cust")
                                    % F.lit(cfg.customer_count) + 1).cast("long"))
        .withColumn("tender_type", d.pick_by_weights(
            ["order_id_ext"], "onl_tender",
            [(n, w) for n, w, _, _, _ in ONLINE_TENDERS]))
        .withColumn("is_cancelled",
                    d.u(["order_id_ext"], "onl_cancel") < F.lit(CANCEL_RATE))
        # per-order promo intensity: 10-30% of its lines carry a promo
        .withColumn("promo_rate",
                    d.u(["order_id_ext"], "onl_prate") * F.lit(0.20) + F.lit(0.10))
        .withColumn("basket_n", basket_n)
        # destination-based tax rate for this order's customer
        .join(cust_rate, "customer_id", "left")
        .withColumn("rate_bps", F.coalesce(F.col("rate_bps"), F.lit(mean_bps)))
    )

    # --- lines: uniform product over the full catalog (no department weights)
    products = dims["dim_products"].select(
        F.col("ID").alias("product_id"), F.col("SalePrice"), F.col("taxability"))
    n_products = products.count()
    products_ranked = products.withColumn(
        "cat_rank", F.row_number().over(Window.orderBy("product_id")))

    qty = d.pick_by_weights(
        ["order_id_ext", "line_num"], "onl_qty",
        [("1", 0.70), ("2", 0.25), ("3", 0.05)]).cast("int")

    promo_idx = (d.h64(["order_id_ext", "line_num"], "onl_pidx") % len(PROMOS)).cast("int")
    promo_code_arr = F.array(*[F.lit(c) for c, _ in PROMOS])
    promo_pct_arr = F.array(*[F.lit(p).cast("long") for _, p in PROMOS])

    # per-line fulfillment: 60% SHIP_FROM_DC / 30% SHIP_FROM_STORE / 10% BOPIS
    mode = d.pick_by_weights(
        ["order_id_ext", "line_num"], "onl_mode",
        [("SHIP_FROM_DC", 0.60), ("SHIP_FROM_STORE", 0.30), ("BOPIS", 0.10)])

    # store tax rate (bps) for destination-based tax on store-fulfilled lines
    store_bps = dims["dim_stores"].select(
        F.col("ID").alias("_snid"),
        F.round(F.col("tax_rate") * 10000).cast("long").alias("_store_bps"))

    lines = (
        orders.select("order_id_ext", "event_ts", "event_date", "is_cancelled",
                      "promo_rate", "basket_n", "rate_bps")
        .withColumn("line_num", F.explode(F.sequence(F.lit(1), F.col("basket_n"))))
        .withColumn("cat_rank", (d.h64(["order_id_ext", "line_num"], "onl_prod")
                                 % F.lit(n_products) + 1).cast("int"))
        .join(products_ranked, "cat_rank")
        .withColumn("quantity", qty.cast("long"))
        .withColumn("unit_cents", F.round(F.col("SalePrice") * 100).cast("long"))
        .withColumn("ext_before", F.col("unit_cents") * F.col("quantity"))
        .withColumn("has_promo", d.u(["order_id_ext", "line_num"], "onl_promo")
                    < F.col("promo_rate"))
        .withColumn("promo_code", F.when(
            F.col("has_promo"), F.element_at(promo_code_arr, promo_idx + 1)))
        .withColumn("promo_pct", F.when(
            F.col("has_promo"), F.element_at(promo_pct_arr, promo_idx + 1))
            .otherwise(F.lit(0).cast("long")))
        # matching 5/10/20% discount, rounded half-up in integer cents
        .withColumn("discount_cents", F.floor(
            (F.col("ext_before") * F.col("promo_pct") + F.lit(50)) / F.lit(100))
            .cast("long"))
        .withColumn("ext_cents", F.col("ext_before") - F.col("discount_cents"))
        # fulfillment + node resolved before tax so the destination rate is known
        .withColumn("fulfillment_mode", mode)
        .withColumn("node_type",
                    F.when(F.col("fulfillment_mode") == "SHIP_FROM_DC", "DC")
                    .otherwise("STORE"))
        .withColumn("node_id", F.when(
            F.col("node_type") == "DC",
            F.element_at(dc_arr, (d.h64(["order_id_ext", "line_num"], "onl_node")
                                  % len(dc_ids) + 1).cast("int")))
            .otherwise(
            F.element_at(st_arr, (d.h64(["order_id_ext", "line_num"], "onl_node")
                                  % len(store_ids) + 1).cast("int"))))
        # destination-based tax (datagen parity): STORE/BOPIS lines use the
        # fulfilling store's rate; DC lines use the customer-geography rate.
        .join(store_bps, F.col("node_id") == F.col("_snid"), "left")
        .withColumn("line_rate_bps", F.when(
            F.col("node_type") == "STORE",
            F.coalesce(F.col("_store_bps"), F.col("rate_bps")))
            .otherwise(F.col("rate_bps")))
        .withColumn("tax_mult", F.when(F.col("taxability") == "TAXABLE", 100)
                    .when(F.col("taxability") == "REDUCED_RATE", 50)
                    .otherwise(0).cast("long"))
        .withColumn("line_tax_cents", F.floor(
            (F.col("ext_cents") * F.col("line_rate_bps") * F.col("tax_mult")
             + F.lit(500_000)) / F.lit(1_000_000)).cast("long"))
        # Backorder a small fraction of shippable (non-BOPIS, non-cancelled)
        # lines: they ship but are not yet delivered (datagen SHIPPED state).
        .withColumn("is_backordered",
                    (F.col("fulfillment_mode") != "BOPIS") & ~F.col("is_cancelled")
                    & (d.u(["order_id_ext", "line_num"], "onl_backorder")
                       < F.lit(BACKORDER_RATE)))
        .withColumn("fulfillment_status",
                    F.when(F.col("is_cancelled"), "CANCELLED")
                    .when(F.col("is_backordered"), "SHIPPED")
                    .otherwise("DELIVERED"))
        .drop("_snid", "_store_bps")
    )

    # --- lifecycle (non-cancelled lines only; cancelled lines keep NULL ts)
    is_bopis = F.col("fulfillment_mode") == "BOPIS"
    u_pick = d.u(["order_id_ext", "line_num"], "onl_pick")
    # BOPIS: picked 4-24h after order; ship modes: 30-240 min
    pick_secs = F.when(
        is_bopis, (F.lit(4 * 3600) + u_pick * F.lit(20 * 3600.0)).cast("long")
    ).otherwise((F.lit(30 * 60) + u_pick * F.lit(210 * 60.0)).cast("long"))
    ship_secs = (F.lit(2 * 3600)
                 + d.u(["order_id_ext", "line_num"], "onl_ship") * F.lit(2 * 3600.0)
                 ).cast("long")
    deliver_secs = (F.lit(1 * 86400)
                    + d.u(["order_id_ext", "line_num"], "onl_dlv") * F.lit(2 * 86400.0)
                    ).cast("long")
    live = ~F.col("is_cancelled")
    lines = (
        lines
        .withColumn("picked_ts", F.when(
            live, F.timestamp_seconds(F.unix_timestamp("event_ts") + pick_secs)))
        # BOPIS never ships: shipped_ts stays NULL
        .withColumn("shipped_ts", F.when(
            live & ~is_bopis,
            F.timestamp_seconds(F.unix_timestamp("picked_ts") + ship_secs)))
        .withColumn("delivered_ts",
                    F.when(F.col("is_backordered"), F.lit(None).cast("timestamp"))
                    .when(live & is_bopis, F.col("picked_ts"))
                    .when(live, F.timestamp_seconds(
                        F.unix_timestamp("shipped_ts") + deliver_secs)))
    )

    # TMDL-bound legacy pandas-index column: hash-derived via legacy_index.
    fact_online_order_lines = (
        lines
        .withColumn("__index_level_0__",
                    legacy_index("order_id_ext", "line_num"))
        .select(
            F.col("order_id_ext").alias("order_id"), "product_id",
            F.col("line_num").cast("long").alias("line_num"), "quantity",
            _fmt(F.col("unit_cents")).alias("unit_price"), "unit_cents",
            _fmt(F.col("ext_cents")).alias("ext_price"), "ext_cents", "promo_code",
            "fulfillment_mode", "fulfillment_status", "node_type", "node_id",
            "picked_ts", "shipped_ts", "delivered_ts", "event_ts", "event_date",
            "__index_level_0__",
        ).select(*column_names("fact_online_order_lines"))
    )

    # --- headers: sum live lines; cancelled orders have financials zeroed
    sums = (
        lines.filter(live)
        .groupBy("order_id_ext")
        .agg(F.sum("ext_cents").alias("subtotal_cents"),
             F.sum("line_tax_cents").alias("tax_cents"))
    )
    headers_base = (
        orders.join(sums, "order_id_ext", "left")
        .withColumn("subtotal_cents",
                    F.coalesce(F.col("subtotal_cents"), F.lit(0)).cast("long"))
        .withColumn("tax_cents", F.coalesce(F.col("tax_cents"), F.lit(0)).cast("long"))
        .withColumn("total_cents", F.col("subtotal_cents") + F.col("tax_cents"))
        .withColumn("payment_method", F.col("tender_type"))
    )
    fact_online_order_headers = headers_base.select(
        "order_id_ext", "customer_id", "subtotal_cents", "tax_cents", "total_cents",
        _fmt(F.col("subtotal_cents")).alias("subtotal_amount"),
        _fmt(F.col("tax_cents")).alias("tax_amount"),
        _fmt(F.col("total_cents")).alias("total_amount"),
        "payment_method", "event_ts", "event_date",
    ).select(*column_names("fact_online_order_headers"))

    # --- payments: one per NON-cancelled order; receipt_id_ext/store_id NULL
    u_dec = d.u(["order_id_ext"], "onl_decline")
    decline_p: Column = F.lit(0.0)
    for name, _, mult, _, _ in ONLINE_TENDERS:
        decline_p = F.when(
            F.col("payment_method") == name, F.lit(BASE_DECLINE * mult)
        ).otherwise(decline_p)
    proc_lo: Column = F.lit(ONLINE_TENDERS[-1][3])
    proc_hi: Column = F.lit(ONLINE_TENDERS[-1][4])
    for name, _, _, lo, hi in ONLINE_TENDERS[:-1]:
        proc_lo = F.when(F.col("payment_method") == name, F.lit(lo)).otherwise(proc_lo)
        proc_hi = F.when(F.col("payment_method") == name, F.lit(hi)).otherwise(proc_hi)
    reason_idx = (d.h64(["order_id_ext"], "onl_reason") % len(DECLINE_REASONS)).cast("int")
    payments = (
        headers_base.filter(~F.col("is_cancelled"))
        .withColumn("receipt_id_ext", F.lit(None).cast("string"))
        .withColumn("store_id", F.lit(None).cast("long"))
        .withColumn("amount_cents", F.col("total_cents"))
        .withColumn("transaction_id", F.concat(
            F.lit("TXN_"), F.unix_timestamp("event_ts").cast("string"), F.lit("_"),
            F.lpad((d.h64(["order_id_ext"], "onl_txn") % 1_000_000).cast("string"),
                   6, "0")))
        .withColumn("status",
                    F.when(u_dec < decline_p, "DECLINED").otherwise("APPROVED"))
        .withColumn("decline_reason", F.when(
            F.col("status") == "DECLINED",
            F.element_at(F.array(*[F.lit(r) for r in DECLINE_REASONS]),
                         reason_idx + 1)))
        .withColumn("processing_time_ms",
                    (proc_lo + d.u(["order_id_ext"], "onl_proc")
                     * (proc_hi - proc_lo)).cast("long"))
        .withColumn("amount", _fmt(F.col("amount_cents")))
        .select(*column_names("fact_payments"))
    )

    return {
        "fact_online_order_headers": fact_online_order_headers,
        "fact_online_order_lines": fact_online_order_lines,
        "payments": payments,
    }

# --- retail_setup/generation/promotions.py ---
"""Promotions fact group, derived purely from receipt lines (no draws).

Semantics (datagen promotions_mixin):

- ``fact_promo_lines``: one row per SALE receipt line that has a non-NULL
  ``promo_code`` and a positive line discount
  (``discount_cents = unit_cents * quantity - ext_cents > 0``). Event fields
  come from the line; ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' +
  promo_code + '-' + line_num``.
- ``fact_promotions``: one row per (receipt_id_ext, promo_code) aggregating
  those lines — discount sums, distinct ``product_count``, ``product_ids`` as
  a comma-joined sorted id list. ``discount_type`` is 'BOGO' for BOGO codes and
  'PERCENTAGE' for all other generated promo codes.
  Store/customer/event fields are joined from the receipt header;
  ``trace_id = 'TRC-PRM-' + receipt_id_ext + '-' + promo_code``.

TMDL contract note: ``fact_promo_lines`` carries dual columns — the
snake_case plan names plus TMDL-bound PascalCase mirrors (ReceiptId,
PromoCode, LineNumber, ProductID, Qty, DiscountAmount, DiscountCents) and the
legacy pandas-index column ``__index_level_0__`` (hash-derived via
legacy_index(receipt_id_ext, promo_code, line_number)). The PascalCase columns
are exact copies of their snake_case twins.

Pure joins/groupBys over the sales group — deterministic, no randomness.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F



def generate_promotions(
    spark: SparkSession,
    sales_group: dict[str, DataFrame],
) -> tuple[DataFrame, DataFrame]:
    """Derive (fact_promotions, fact_promo_lines) from the sales group."""

    sale_ids = (
        sales_group["fact_receipts"]
        .filter(F.col("receipt_type") == "SALE")
        .select("receipt_id_ext")
    )

    # --- promo lines: discounted SALE lines with a promo code
    lines = (
        sales_group["fact_receipt_lines"]
        .join(sale_ids, "receipt_id_ext")
        .filter(F.col("promo_code").isNotNull())
        .withColumn(
            "discount_cents",
            F.col("unit_cents") * F.col("quantity") - F.col("ext_cents"),
        )
        .filter(F.col("discount_cents") > 0)
        .withColumn("line_number", F.col("line_num").cast("long"))
        .withColumn("quantity", F.col("quantity").cast("long"))
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        .withColumn(
            "trace_id",
            F.concat(
                F.lit("TRC-PRM-"), F.col("receipt_id_ext"), F.lit("-"),
                F.col("promo_code"), F.lit("-"),
                F.col("line_number").cast("string"),
            ),
        )
    )

    promo_lines = (
        lines
        # TMDL-bound PascalCase mirrors of the snake_case columns
        .withColumn("ReceiptId", F.col("receipt_id_ext"))
        .withColumn("PromoCode", F.col("promo_code"))
        .withColumn("LineNumber", F.col("line_number"))
        .withColumn("ProductID", F.col("product_id"))
        .withColumn("Qty", F.col("quantity"))
        .withColumn("DiscountAmount", F.col("discount_amount"))
        .withColumn("DiscountCents", F.col("discount_cents"))
        # Legacy pandas-index column bound by the semantic model
        .withColumn("__index_level_0__",
                    legacy_index("receipt_id_ext", "promo_code", "line_number"))
        .select(*column_names("fact_promo_lines"))
    )

    # --- promotions: aggregate per (receipt_id_ext, promo_code)
    agg = lines.groupBy("receipt_id_ext", "promo_code").agg(
        F.sum("discount_cents").alias("discount_cents"),
        F.countDistinct("product_id").alias("product_count"),
        F.concat_ws(",", F.sort_array(F.collect_set("product_id"))).alias("product_ids"),
    )

    headers = sales_group["fact_receipts"].select(
        "receipt_id_ext", "event_ts", "event_date", "store_id", "customer_id"
    )
    promotions = (
        agg
        .join(headers, "receipt_id_ext")
        .withColumn("discount_amount", _fmt(F.col("discount_cents")))
        # BOGO codes are buy-one-get-one; everything else is a percentage off.
        .withColumn("discount_type",
                    F.when(F.col("promo_code").startswith("BOGO"), F.lit("BOGO"))
                    .otherwise(F.lit("PERCENTAGE")))
        .withColumn(
            "trace_id",
            F.concat(F.lit("TRC-PRM-"), F.col("receipt_id_ext"), F.lit("-"),
                     F.col("promo_code")),
        )
        .select(*column_names("fact_promotions"))
    )

    return promotions, promo_lines

# --- retail_setup/generation/marketing.py ---
"""Marketing impressions fact, Spark-native.

Ports datagen's marketing_campaign.py constants: four campaign archetypes,
each with its own channel mix and base impressions/day. Daily volumes are
scaled by `cfg.store_count / 86` — 86 is the legacy datagen fleet size, so a
fleet of 86 stores reproduces datagen's nominal volumes. `flash_sale` runs
only ~1 day in 7 (a per-day uniform gate, u < 1/7).

Per impression: channel uniform over the archetype's channels; device
MOBILE/DESKTOP/TABLET at 60/30/10 with cost multipliers 1.2/0.8/0.9; cost a
uniform draw within the channel's dollar band times the device multiplier;
a uniformly-assigned customer supplies `customer_ad_id` (dim_customers.AdId),
with `customer_id` populated for ~5% of impressions (CRM match) else NULL.
All draws via `runtime.seeded_draws`, so output is deterministic for a
(config, seed) pair regardless of cluster shape.

Schema note: fact_marketing is TMDL-bound with DUAL columns — PascalCase
`CustomerId`/`CostCents` (the TMDL sourceColumns) are exact mirrors of the
snake_case plan columns `customer_id`/`cost_cents`, and `__index_level_0__`
is the legacy pandas index (hash-derived via legacy_index).
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


# (archetype, channels, base impressions/day at the legacy 86-store fleet,
#  campaign duration in days — datagen MarketingCampaignSimulator templates)
ARCHETYPES: list[tuple[str, list[str], int, int]] = [
    ("seasonal_sale", ["FACEBOOK", "GOOGLE", "EMAIL"], 1000, 7),
    ("product_launch", ["INSTAGRAM", "YOUTUBE", "DISPLAY"], 2000, 14),
    ("loyalty_program", ["EMAIL", "SOCIAL"], 500, 30),
    ("flash_sale", ["SOCIAL", "SEARCH"], 5000, 1),  # runs ~1 day in 7
]

# channel -> (lo, hi) cost-per-impression band in dollars (uniform draw)
CHANNEL_COSTS: dict[str, tuple[float, float]] = {
    "EMAIL": (0.005, 0.05),
    "DISPLAY": (0.10, 0.50),
    "SOCIAL": (0.08, 0.25),
    "SEARCH": (0.15, 0.75),
    "FACEBOOK": (0.10, 0.40),
    "GOOGLE": (0.25, 1.50),
    "INSTAGRAM": (0.08, 0.35),
    "YOUTUBE": (0.30, 1.50),
}

LEGACY_FLEET_SIZE = 86  # datagen's store count; volumes scale store_count/86

DEVICE_WEIGHTS = [("MOBILE", 0.6), ("DESKTOP", 0.3), ("TABLET", 0.1)]
DEVICE_MULTIPLIERS = {"MOBILE": 1.2, "DESKTOP": 0.8, "TABLET": 0.9}


def generate_marketing(
    spark: SparkSession,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> DataFrame:
    """Ad-impression facts: day x archetype grid exploded to impressions."""

    d = seeded_draws(cfg.seed)
    scale = cfg.store_count / LEGACY_FLEET_SIZE

    # --- day x archetype grid (driver-side; days x 4 rows)
    rows = [
        (idx + 1, name, channels, base, dur)
        for idx, (name, channels, base, dur) in enumerate(ARCHETYPES)
    ]
    grid = spark.createDataFrame(
        [
            (day_offset, idx, name, channels, base, dur)
            for day_offset in range((cfg.end_date - cfg.start_date).days + 1)
            for (idx, name, channels, base, dur) in rows
        ],
        "day_offset int, archetype_idx int, archetype string, "
        "channels array<string>, base_impressions int, duration int",
    ).withColumn(
        "day", F.date_add(F.lit(cfg.start_date), F.col("day_offset"))
    ).withColumn(
        # campaigns span their archetype duration: the day's impressions belong
        # to the campaign window [start + k*duration, ...) it falls in.
        "campaign_start",
        F.date_add(F.lit(cfg.start_date),
                   (F.col("day_offset") / F.col("duration")).cast("int")
                   * F.col("duration")),
    )

    # flash_sale runs ~1 day in 7 (uniform gate per day)
    grid = grid.filter(
        (F.col("archetype") != "flash_sale")
        | (d.u([F.col("day")], "mk_flash_gate") < F.lit(1.0 / 7.0))
    )

    # per-day impression count: clamped normal around the scaled base
    lam = F.col("base_impressions") * F.lit(scale)
    n_imps = F.greatest(
        F.lit(1),
        F.round(lam + d.gauss(["day", "archetype"], "mk_n") * F.sqrt(lam)),
    ).cast("int")
    grid = grid.withColumn("n_impressions", n_imps)

    # --- explode to impressions; build IDs
    imps = (
        grid
        .withColumn("seq", F.explode(F.sequence(F.lit(1), F.col("n_impressions"))))
        .withColumn(
            # campaign_id keyed on the campaign window so it spans multiple days
            "campaign_id",
            F.concat(
                F.lit("CAMP"),
                F.date_format("campaign_start", "yyyyMMdd"),
                F.lpad(F.col("archetype_idx").cast("string"), 2, "0"),
            ),
        )
        .withColumn(
            # impression id stays day-unique (campaign_id now repeats across days)
            "impression_id_ext",
            F.concat(
                F.lit("IMP"), F.date_format("day", "yyyyMMdd"),
                F.lpad(F.col("archetype_idx").cast("string"), 2, "0"),
                F.lpad(F.col("seq").cast("string"), 7, "0"),
            ),
        )
        # drops the 'IMP' prefix: CREAT + day + archetype + seq
        .withColumn(
            "creative_id",
            F.concat(F.lit("CREAT"), F.substring("impression_id_ext", 4, 30)),
        )
    )

    key = [F.col("impression_id_ext")]

    # --- channel uniform over the archetype's channels
    ch_idx = F.floor(d.u(key, "mk_channel") * F.size("channels")).cast("int")
    imps = imps.withColumn("channel", F.element_at("channels", ch_idx + 1))

    # --- device pick (60/30/10) and cost multiplier
    imps = imps.withColumn(
        "device", d.pick_by_weights(key, "mk_device", DEVICE_WEIGHTS)
    )
    mult = None
    for device, m in DEVICE_MULTIPLIERS.items():
        cond = F.col("device") == device
        mult = F.when(cond, m) if mult is None else mult.when(cond, m)
    imps = imps.withColumn("device_mult", mult)

    # --- cost: uniform within the channel band, times the device multiplier
    lo, hi = None, None
    for channel, (c_lo, c_hi) in CHANNEL_COSTS.items():
        cond = F.col("channel") == channel
        lo = F.when(cond, c_lo) if lo is None else lo.when(cond, c_lo)
        hi = F.when(cond, c_hi) if hi is None else hi.when(cond, c_hi)
    cost_dollars = (lo + d.u(key, "mk_cost") * (hi - lo)) * F.col("device_mult")
    imps = (
        imps
        .withColumn("cost_cents", F.round(cost_dollars * 100).cast("long"))
        .withColumn("cost", F.format_string("%.2f", cost_dollars))
    )

    # --- uniform customer per impression; ~5% CRM-matched (customer_id set)
    imps = imps.withColumn(
        "customer_pick",
        (d.h64(key, "mk_customer") % F.lit(cfg.customer_count) + 1).cast("long"),
    )
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_pick"), F.col("AdId").alias("customer_ad_id")
    )
    imps = imps.join(customers, "customer_pick", "left")
    imps = imps.withColumn(
        "customer_id",
        F.when(
            d.u(key, "mk_crm") < F.lit(0.05),
            F.col("customer_pick").cast("double"),
        ),
    )

    # --- event_ts uniform within the day
    day_ts = F.unix_timestamp(F.col("day").cast("timestamp"))
    event_ts = F.timestamp_seconds(
        day_ts + F.floor(d.u(key, "mk_ts") * F.lit(86400))
    )

    out = imps.select(
        event_ts.alias("event_ts"),
        F.concat(F.lit("TRC-MKT-"), F.col("impression_id_ext")).alias("trace_id"),
        "channel",
        "campaign_id",
        "creative_id",
        "customer_ad_id",
        "customer_id",
        "cost_cents",
        # TMDL-bound PascalCase duals: exact mirrors of the snake_case columns
        F.col("customer_id").alias("CustomerId"),
        F.col("cost_cents").alias("CostCents"),
        "impression_id_ext",
        "cost",
        "device",
        F.col("day").alias("event_date"),
    ).withColumn(
        "__index_level_0__",
        legacy_index("impression_id_ext"),
    )
    return out.select(*column_names("fact_marketing"))

# --- retail_setup/generation/sensors.py ---
"""BLE pings + customer zone changes, Spark-native.

Derives from SALE receipts (one BLE visit per receipt):
- 30% "known" visits: customer_ble_id from dim_customers.BLEId joined on the
  receipt's customer_id; customer_id (double) carried through.
- 70% anonymous: customer_ble_id = 'ANON-<store_id>-<h64 % 900000 + 100000>';
  customer_id NULL.

Per visit, 2-5 zones are selected (uniform draw over n_zones, then zone selection
without replacement via per-zone draw + window rank over the 5-zone array using
posexplode). Each zone gets 2-5 pings. Ping event_ts = receipt event_ts +
offset minutes ((zone_rank - 1) * 7 + ping_seq + u) - 15, spreading the visit
±15 min around the receipt timestamp.

Deviation from spec's "pandas-UDF island" suggestion: uses Spark-native window
functions instead. The correlation between zones is mild enough for this approach
and avoids applyInPandas/F.rand() (see self-review checklist).

Schema notes (TMDL arbiter results):
- fact_ble_pings: BOTH customer_id (double, snake) AND CustomerId (double,
  TMDL-bound) with mirrored values; __index_level_0__ (long) = hash-derived
  via legacy_index(trace_id).
- fact_customer_zone_changes: BOTH snake_case (store_id, customer_ble_id,
  from_zone, to_zone) AND PascalCase (StoreID, CustomerBLEId, FromZone, ToZone)
  with mirrored values; __index_level_0__ = legacy_index(trace_id).
"""

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F


# BLE zone names (5 total); zone selection is without replacement
BLE_ZONES = ["ENTRANCE", "ELECTRONICS", "GROCERY", "CLOTHING", "CHECKOUT"]
_N_ZONES = len(BLE_ZONES)  # 5


def generate_ble(
    spark: SparkSession,
    receipts: DataFrame,
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> tuple[DataFrame, DataFrame]:
    """Generate BLE ping and zone-change facts from SALE receipts.

    Returns:
        (pings_df, zone_changes_df)
    """
    d = seeded_draws(cfg.seed)

    # --- filter to SALE receipts (defensively; receipts may already be SALE-only)
    visits = receipts.filter(F.col("receipt_type") == "SALE")

    # --- 30% known (u < 0.3): join dim_customers for BLEId; 70% anonymous
    customers = dims["dim_customers"].select(
        F.col("ID").alias("customer_id_long"),
        F.col("BLEId").alias("ble_id_from_dim"),
    )

    # Receipt customer_id may be stored as a non-double type; cast for join
    visits = visits.withColumn("_cust_key", F.col("customer_id").cast("long"))
    visits = visits.join(
        customers,
        visits._cust_key == customers.customer_id_long,
        "left",
    )

    # known flag: u(receipt_id_ext, "known") < 0.3
    visits = visits.withColumn(
        "_known", d.u(["receipt_id_ext"], "known") < F.lit(0.3)
    )

    # customer_ble_id and customer_id (double)
    visits = visits.withColumn(
        "customer_ble_id",
        F.when(
            F.col("_known") & F.col("ble_id_from_dim").isNotNull(),
            F.col("ble_id_from_dim"),
        ).otherwise(
            F.concat(
                F.lit("ANON-"),
                F.col("store_id").cast("string"),
                F.lit("-"),
                (d.h64(["receipt_id_ext"], "anon_id") % F.lit(900000) + F.lit(100000))
                    .cast("string"),
            )
        ),
    ).withColumn(
        "visit_customer_id",
        F.when(
            F.col("_known") & F.col("ble_id_from_dim").isNotNull(),
            F.col("customer_id").cast("double"),
        ).cast("double"),
    )

    # --- draw n_zones per visit (2-5 inclusive)
    # u_zones in [0,1) → floor(u * 4) + 2 = 2..5
    visits = visits.withColumn(
        "n_zones",
        (F.floor(d.u(["receipt_id_ext"], "nz") * F.lit(4)) + F.lit(2)).cast("int"),
    )

    # --- zone selection without replacement via posexplode + per-zone draw + rank
    # Explode the 5-zone array to get (pos, zone) rows per visit
    zone_array = F.array(*[F.lit(z) for z in BLE_ZONES])
    visits_with_zones = visits.select(
        "receipt_id_ext", "store_id", "event_ts", "event_date",
        "customer_ble_id", "visit_customer_id", "n_zones",
        F.posexplode(zone_array).alias("zone_pos", "zone"),
    )

    # Assign a random draw per (receipt, zone) to shuffle zone order
    visits_with_zones = visits_with_zones.withColumn(
        "zone_draw",
        d.u(
            [F.col("receipt_id_ext"), F.col("zone_pos").cast("string")],
            "z_shuffle",
        ),
    )

    # Rank zones by draw within each visit (rank 1 = first selected zone)
    w_visit = Window.partitionBy("receipt_id_ext").orderBy("zone_draw")
    visits_with_zones = visits_with_zones.withColumn(
        "zone_rank", F.row_number().over(w_visit).cast("int")
    )

    # Keep only the top n_zones ranked zones (zone_rank <= n_zones)
    selected_zones = visits_with_zones.filter(F.col("zone_rank") <= F.col("n_zones"))

    # --- draw n_pings per zone (2-5 inclusive)
    selected_zones = selected_zones.withColumn(
        "n_pings",
        (
            F.floor(
                d.u(
                    [F.col("receipt_id_ext"), F.col("zone_rank").cast("string")],
                    "np",
                ) * F.lit(4)
            ) + F.lit(2)
        ).cast("int"),
    )

    # --- explode to individual pings
    pings = selected_zones.select(
        "receipt_id_ext", "store_id", "event_ts", "event_date",
        "customer_ble_id", "visit_customer_id", "zone_rank", "zone", "n_pings",
        F.explode(F.sequence(F.lit(1), F.col("n_pings"))).alias("ping_seq"),
    )

    # Global sequence for trace_id uniqueness: row number over (receipt, zone_rank, ping_seq)
    w_ping_order = Window.partitionBy("receipt_id_ext").orderBy("zone_rank", "ping_seq")
    pings = pings.withColumn(
        "ping_visit_seq", F.row_number().over(w_ping_order).cast("long")
    )

    ping_key = [
        F.col("receipt_id_ext"),
        F.col("zone_rank").cast("string"),
        F.col("ping_seq").cast("string"),
    ]

    # offset_min = (zone_rank - 1) * 7 + ping_seq + u - 15
    offset_min = (
        (F.col("zone_rank") - F.lit(1)) * F.lit(7)
        + F.col("ping_seq")
        + d.u(ping_key, "ts_offset")
        - F.lit(15.0)
    )
    ping_ts = F.timestamp_seconds(
        F.unix_timestamp("event_ts") + (offset_min * F.lit(60.0)).cast("long")
    )

    # rssi uniform [-80, -29] (integer long)
    rssi = (
        F.floor(d.u(ping_key, "rssi") * F.lit(52.0)).cast("long") - F.lit(80)
    )

    pings = pings.withColumn("ping_ts", ping_ts).withColumn("rssi", rssi)
    pings = pings.withColumn(
        "trace_id",
        F.concat(
            F.lit("TRC-BLE-"),
            F.col("receipt_id_ext"),
            F.lit("-"),
            F.col("ping_visit_seq").cast("string"),
        ),
    ).withColumn(
        "beacon_id",
        F.format_string("BEACON_%03d_%s", F.col("store_id"), F.col("zone")),
    )

    # __index_level_0__ = hash-derived via legacy_index(trace_id)
    pings_out = pings.select(
        F.col("ping_ts").alias("event_ts"),
        "trace_id",
        F.col("store_id").cast("long").alias("store_id"),
        "beacon_id",
        "customer_ble_id",
        F.col("visit_customer_id").alias("customer_id"),
        # TMDL-bound PascalCase mirror
        F.col("visit_customer_id").alias("CustomerId"),
        F.col("rssi").cast("long").alias("rssi"),
        "zone",
        F.to_date("ping_ts").alias("event_date"),
    ).withColumn(
        "__index_level_0__",
        legacy_index("trace_id"),
    )
    pings_df = pings_out.select(*column_names("fact_ble_pings"))

    # --- zone changes: window per (store_id, customer_ble_id, receipt_id_ext)
    # ordered by ping event_ts; lag(zone); keep where zone changed
    w_visit_ts = Window.partitionBy(
        "store_id", "customer_ble_id", "receipt_id_ext"
    ).orderBy("ping_ts")

    zc_base = pings.select(
        "store_id", "customer_ble_id", "receipt_id_ext", "ping_ts", "zone",
    ).withColumn("prev_zone", F.lag("zone").over(w_visit_ts))

    zc_base = zc_base.filter(
        F.col("prev_zone").isNotNull() & (F.col("zone") != F.col("prev_zone"))
    )

    # trace_id sequence per visit
    w_zc_seq = Window.partitionBy("store_id", "customer_ble_id", "receipt_id_ext").orderBy("ping_ts")
    zc_base = zc_base.withColumn("zc_seq", F.row_number().over(w_zc_seq).cast("long"))

    zc_out = zc_base.select(
        F.col("ping_ts").alias("event_ts"),
        F.concat(
            F.lit("TRC-ZC-"),
            F.col("receipt_id_ext"),
            F.lit("-"),
            F.col("zc_seq").cast("string"),
        ).alias("trace_id"),
        # snake_case (extras)
        F.col("store_id").cast("long").alias("store_id"),
        F.col("customer_ble_id").alias("customer_ble_id"),
        F.col("prev_zone").alias("from_zone"),
        F.col("zone").alias("to_zone"),
        # TMDL-bound PascalCase mirrors
        F.col("store_id").cast("long").alias("StoreID"),
        F.col("customer_ble_id").alias("CustomerBLEId"),
        F.col("prev_zone").alias("FromZone"),
        F.col("zone").alias("ToZone"),
        F.to_date("ping_ts").alias("event_date"),
    ).withColumn(
        "__index_level_0__",
        legacy_index("trace_id"),
    )
    zc_df = zc_out.select(*column_names("fact_customer_zone_changes"))

    return pings_df, zc_df

# --- retail_setup/generation/inventory_balances.py ---
"""Balance + stockout helpers for the inventory chain (Plan 2b Task 9).

Split out of ``inventory.py`` per the plan's ~400-line guidance. Covers
stages 7-8: day-0 INITIAL seed txns plus the running-balance window, and the
balance-crossing stockout extraction. Shared draw/column primitives used by
both modules live here to keep the import direction one-way
(``inventory`` -> ``inventory_balances``).
"""

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Canonical column layout for raw (pre-balance) inventory txn streams.
TXN_COLS = ["node_id", "product_id", "quantity", "txn_type", "source",
            "event_ts", "event_date", "trace_id"]


def draw_int(u: Column, lo: int, hi: int) -> Column:
    """Uniform integer draw in [lo, hi] from a [0,1) uniform column."""
    return (F.lit(lo) + F.floor(u * F.lit(hi - lo + 1))).cast("long")


# ---------------------------------------------------------------------------
# Stage 7: initial-stock seeds + running balances
# ---------------------------------------------------------------------------

def with_balances(txns: DataFrame, lo: int, hi: int, tag: str,
                  d: seeded_draws, cfg: GenerationConfig) -> DataFrame:
    """Fold a day-0 INITIAL seed txn per (node, product) into the stream and
    compute the running balance ordered by (event_ts, trace_id). Negative
    balances are not clamped — they become stockout signals."""
    seeds = (txns.select("node_id", "product_id").distinct()
             .withColumn("quantity",
                         draw_int(d.u(["node_id", "product_id"],
                                      f"seed-stock-{tag}"), lo, hi))
             .withColumn("txn_type", F.lit("INITIAL"))
             .withColumn("source", F.lit("SEED"))
             # String-built timestamp (session-TZ semantics) to match every
             # other event_ts in the chain; a naive Python datetime literal
             # would shift with the driver's local timezone.
             .withColumn("event_ts", F.to_timestamp(
                 F.lit(f"{cfg.start_date.isoformat()} 00:00:00")))
             .withColumn("event_date", F.lit(cfg.start_date).cast("date"))
             .withColumn("trace_id", F.concat(
                 F.lit(f"TRC-INIT-{tag}-"), F.col("node_id").cast("string"),
                 F.lit("-"), F.col("product_id").cast("string")))
             .select(*TXN_COLS))
    run_w = (Window.partitionBy("node_id", "product_id")
             .orderBy("event_ts", "trace_id")
             .rowsBetween(Window.unboundedPreceding, Window.currentRow))
    return (txns.unionByName(seeds)
            .withColumn("balance", F.sum("quantity").over(run_w).cast("long")))


# ---------------------------------------------------------------------------
# Stage 8: stockouts
# ---------------------------------------------------------------------------

def stockouts(balanced: DataFrame, tag: str, node_as: str) -> DataFrame:
    """Balance crossings to <=0 (previous balance > 0); deduped to one per
    (node, product, day). ``node_as`` is 'StoreID' or 'DCID' — the other
    contract column stays NULL (double, per the TMDL contract)."""
    order_w = Window.partitionBy("node_id", "product_id").orderBy(
        "event_ts", "trace_id")
    day_w = Window.partitionBy("node_id", "product_id", "event_date").orderBy(
        "event_ts", "trace_id")
    other = "DCID" if node_as == "StoreID" else "StoreID"
    return (balanced
            .withColumn("_prev", F.lag("balance").over(order_w))
            .filter((F.col("balance") <= 0) & (F.col("_prev") > 0))
            .withColumn("_dup", F.row_number().over(day_w))
            .filter(F.col("_dup") == 1)
            .select(
                "event_ts",
                F.concat(F.lit(f"TRC-SO-{tag}-"),
                         F.col("node_id").cast("string"), F.lit("-"),
                         F.col("product_id").cast("string"), F.lit("-"),
                         F.date_format("event_date", "yyyyMMdd"))
                .alias("trace_id"),
                F.col("node_id").cast("double").alias(node_as),
                F.lit(None).cast("double").alias(other),
                F.col("product_id").alias("ProductID"),
                F.abs("quantity").cast("long").alias("LastKnownQuantity"),
                "event_date",
            ))

# --- retail_setup/generation/inventory.py ---
"""Inventory & logistics chain, Spark-native (Plan 2b Task 9).

Eight internal stages produce six fact tables from the sales/returns groups:

1. Store SALE txns mirror ``fact_receipt_lines`` 1:1 (negative quantity);
   RETURN add-backs mirror return lines (positive quantity).
2. Reorders: per (store, day), the day's top-5 demanded products gated at
   ``u < 0.4`` each emit one reorder. Stores route to the nearest DC by
   state/region and high-volume store-days split across truck-capacity legs.
3. Shipments: one per (store, day, leg) with reorders. The truck is chosen
   round-robin by day number from that DC's assigned trucks in ``dim_trucks``
   (``DCID == dc``); a DC with no assigned trucks falls back to the shared
   pool trucks (``DCID IS NULL``). Six
   ``fact_truck_moves`` status rows per shipment: SCHEDULED day 23:30,
   LOADING next-day 04:00, IN_TRANSIT 06:00 (departure), ARRIVED 06:00+travel
   (= eta), UNLOADING eta+0.25h, COMPLETED eta+unload (= etd). eta/etd are
   populated on all six rows (known at scheduling); departure_time and
   actual_unload_duration only on COMPLETED.
4. Truck inventory: LOAD at the DC (LOADING ts) + UNLOAD at the store
   (UNLOADING ts) per shipment product, qty = reorder_quantity.
5. DC txns: supplier INBOUND_SHIPMENT per (dc, day) — 1-3 deliveries x 5
   uniform product picks x qty 50-500 at day 08:00 (time is a documented
   choice; the plan leaves it open) — plus OUTBOUND_SHIPMENT rows mirroring
   truck LOADs (negative reorder_quantity, source = shipment_id).
6. Store INBOUND_SHIPMENT rows mirroring truck UNLOADs (positive quantity,
   source = shipment_id, UNLOADING ts).
7. Balances: a day-0 INITIAL seed txn per (node, product) seen in that
   node's stream (store 40-120, DC 500-2000, source 'SEED'), then a running
   ``sum(quantity)`` window ordered by (event_ts, trace_id). Negatives are
   not clamped — they become stockout signals.
8. Stockouts: txns where the running balance crosses to <= 0 (previous
   balance > 0), deduped to one per (node, product, day). StoreID/DCID are
   mutually exclusive doubles per the TMDL contract.

All randomness flows through ``runtime.seeded_draws`` so output is
deterministic per (config, seed). Stage 7-8 helpers (balances, stockouts)
live in ``inventory_balances.py`` per the plan's ~400-line split guidance.
"""

from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


_REORDER_TOP_N = 5
_REORDER_GATE = 0.4
# Fraction of returned units restocked to store on-hand; the rest are destroyed
# or returned-to-vendor and never re-enter inventory (datagen returns
# disposition: food destroyed, non-food ~40% restock).
_RETURN_RESTOCK_RATE = 0.5


def _at(day: Column, hhmmss: str) -> Column:
    """Timestamp at a fixed wall-clock time on a date column."""
    return F.to_timestamp(F.concat(day.cast("string"), F.lit(f" {hhmmss}")))


def _plus_hours(ts: Column, hours: Column) -> Column:
    return F.timestamp_seconds(
        F.unix_timestamp(ts) + (hours * F.lit(3600.0)).cast("long"))


def _with_index(df: DataFrame, table: str) -> DataFrame:
    """Add the TMDL-bound __index_level_0__ column and apply the contract order."""
    return (df.withColumn("__index_level_0__", legacy_index("trace_id"))
            .select(*column_names(table)))


# ---------------------------------------------------------------------------
# Stage 1: store demand (SALE mirrors + RETURN add-backs)
# ---------------------------------------------------------------------------

def _sale_txns(sales: dict[str, DataFrame], rets: dict[str, DataFrame],
               d: seeded_draws) -> DataFrame:
    def _lines_to_txns(group: dict[str, DataFrame], txn_type: str,
                       source: Column, restock_gate: bool = False) -> DataFrame:
        hdr = group["fact_receipts"].select("receipt_id_ext", "store_id")
        # SALE lines carry positive qty -> negate; RETURN lines carry negative
        # qty -> negate back to a positive add-back. Both are just -quantity.
        base = group["fact_receipt_lines"].join(hdr, "receipt_id_ext")
        if restock_gate:
            # Only a fraction of returned units re-enter store on-hand; the rest
            # are destroyed / returned-to-vendor (datagen returns disposition).
            base = base.filter(
                d.u([F.col("receipt_id_ext"), F.col("line_num").cast("string")],
                    "restock") < F.lit(_RETURN_RESTOCK_RATE))
        return base.select(
            F.col("store_id").alias("node_id"),
            "product_id",
            (-F.col("quantity")).cast("long").alias("quantity"),
            F.lit(txn_type).alias("txn_type"),
            source.alias("source"),
            "event_ts", "event_date",
            F.concat(F.lit("TRC-INV-"), F.col("receipt_id_ext"),
                     F.lit("-"), F.col("line_num").cast("string"))
            .alias("trace_id"),
        )

    sale = _lines_to_txns(sales, "SALE", F.lit("CUSTOMER_PURCHASE"))
    ret = _lines_to_txns(rets, "RETURN", F.col("receipt_id_ext"), restock_gate=True)
    return sale.unionByName(ret).select(*TXN_COLS)


# ---------------------------------------------------------------------------
# Stage 2: reorders
# ---------------------------------------------------------------------------

def _store_dc_map(dims: dict[str, DataFrame]) -> DataFrame:
    """Route each store to its nearest DC by geography (same state > same region
    > lowest dc_id), restoring datagen's geography-aware assignment in place of a
    ``store_id % dc_count`` modulo. Returns one (store_id, dc_id) row per store."""
    geo = dims["dim_geographies"].select(
        F.col("ID").alias("geo_id"), "State", "Region")
    stores = (dims["dim_stores"].select(F.col("ID").alias("store_id"), "GeographyID")
              .join(geo, F.col("GeographyID") == F.col("geo_id"))
              .select("store_id", F.col("State").alias("s_state"),
                      F.col("Region").alias("s_region")))
    dcs = (dims["dim_distribution_centers"]
           .select(F.col("ID").alias("dc_id"), "GeographyID")
           .join(geo, F.col("GeographyID") == F.col("geo_id"))
           .select("dc_id", F.col("State").alias("d_state"),
                   F.col("Region").alias("d_region")))
    scored = stores.crossJoin(dcs).withColumn(
        "_score",
        F.when(F.col("s_state") == F.col("d_state"), 0)
        .when(F.col("s_region") == F.col("d_region"), 1)
        .otherwise(2))
    nearest = Window.partitionBy("store_id").orderBy("_score", "dc_id")
    return (scored.withColumn("_r", F.row_number().over(nearest))
            .filter(F.col("_r") == 1)
            .select("store_id", F.col("dc_id").cast("long").alias("dc_id")))


def _reorders(store_txns: DataFrame, store_dc: DataFrame, d: seeded_draws,
              cfg: GenerationConfig) -> DataFrame:
    demand = (store_txns.filter(F.col("txn_type") == "SALE")
              .groupBy(F.col("node_id").alias("store_id"),
                       "event_date", "product_id")
              .agg(F.sum(-F.col("quantity")).alias("demand")))
    top_w = Window.partitionBy("store_id", "event_date").orderBy(
        F.desc("demand"), "product_id")
    keys = ["store_id", "event_date", "product_id"]
    return (demand
            .withColumn("_rank", F.row_number().over(top_w))
            .filter(F.col("_rank") <= _REORDER_TOP_N)
            .filter(d.u(keys, "reorder-gate") < F.lit(_REORDER_GATE))
            .withColumn("reorder_point", draw_int(d.u(keys, "reorder-point"), 5, 20))
            .withColumn("current_quantity", F.greatest(
                F.lit(0).cast("long"),
                F.col("reorder_point") - F.col("demand")))
            .withColumn("reorder_quantity", draw_int(d.u(keys, "reorder-qty"), 50, 200))
            # split a store-day's products across truck legs by capacity: each
            # leg holds <= truck_capacity units (datagen multi-truck shipments).
            .withColumn("_cum_qty", F.sum("reorder_quantity").over(
                Window.partitionBy("store_id", "event_date").orderBy("product_id")
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)))
            .withColumn("leg", F.floor(
                (F.col("_cum_qty") - F.col("reorder_quantity"))
                / F.lit(cfg.truck_capacity)).cast("long"))
            .withColumn("_deficit_pct",
                        (F.col("reorder_point") - F.col("current_quantity"))
                        / F.col("reorder_point") * F.lit(100.0))
            .withColumn("priority",
                        F.when(F.col("_deficit_pct") >= 50, "URGENT")
                        .when(F.col("_deficit_pct") >= 25, "HIGH")
                        .otherwise("NORMAL"))
            .withColumn("event_ts", _at(F.col("event_date"), "23:00:00"))
            # Geography-aware store->DC routing (nearest DC by state/region).
            .join(store_dc, "store_id")
            .withColumn("trace_id", F.concat(
                F.lit("TRC-RO-"), F.col("store_id").cast("string"), F.lit("-"),
                F.col("event_date").cast("string"), F.lit("-"),
                F.col("product_id").cast("string")))
            .drop("_rank", "_deficit_pct", "demand", "_cum_qty"))


# ---------------------------------------------------------------------------
# Stage 3: shipments (one per store-day with reorders) + truck assignment
# ---------------------------------------------------------------------------

def _truck_lookup(spark: SparkSession, dims: dict[str, DataFrame],
                  cfg: GenerationConfig) -> DataFrame:
    """(dc_id, slot, truck_id, n_trucks) rows for round-robin-by-day joins."""
    trucks = dims["dim_trucks"].select("ID", "DCID").collect()
    assigned: dict[int, list[int]] = {}
    pool: list[int] = []
    for r in trucks:
        if r.DCID is None:
            pool.append(int(r.ID))
        else:
            assigned.setdefault(int(r.DCID), []).append(int(r.ID))
    rows = []
    for dc in range(1, cfg.dc_count + 1):
        # Fall back to pool trucks for DCs with no assigned trucks (documented).
        fleet = sorted(assigned.get(dc) or pool)
        rows.extend((dc, slot, tid, len(fleet)) for slot, tid in enumerate(fleet))
    return spark.createDataFrame(
        rows, "dc_id long, slot long, truck_id long, n_trucks long")


def _shipments(spark: SparkSession, reorders: DataFrame,
               dims: dict[str, DataFrame], d: seeded_draws,
               cfg: GenerationConfig) -> DataFrame:
    """One row per shipment leg with truck assignment and the full timing model.

    A store-day's reorders are split into legs of <= ``truck_capacity`` units;
    each leg is its own shipment (own truck + 6-status lifecycle), staggered 30
    min apart. At demo scale every store-day fits one leg, so this reduces to a
    single shipment identical to the pre-split behaviour.
    """
    keys = ["store_id", "event_date"]
    base = (reorders.select("store_id", "event_date", "dc_id", "leg").distinct()
            .withColumn("shipment_id", F.concat(
                F.lit("SHIP"), F.date_format("event_date", "yyyyMMdd"),
                F.lpad(F.col("dc_id").cast("string"), 2, "0"),
                F.lpad(F.col("store_id").cast("string"), 3, "0"),
                F.lpad(F.col("leg").cast("string"), 2, "0")))
            .withColumn("_day_num", F.datediff(
                F.col("event_date"), F.lit(cfg.start_date))))
    lookup = _truck_lookup(spark, dims, cfg)
    sizes = lookup.select("dc_id", "n_trucks").distinct()
    timed = (base
             .join(sizes, "dc_id")
             .withColumn("slot", F.pmod(
                 F.col("_day_num") + F.col("leg"), F.col("n_trucks")))
             .join(lookup, ["dc_id", "slot", "n_trucks"])
             .withColumn("_travel_h",
                         F.lit(2.0) + d.u(keys, "ship-travel") * F.lit(10.0))
             .withColumn("_unload_h",
                         F.lit(0.5) + d.u(keys, "ship-unload") * F.lit(1.5))
             # legs depart 30 min apart
             .withColumn("scheduled_ts", _plus_hours(
                 _at(F.col("event_date"), "23:30:00"), F.col("leg") * F.lit(0.5)))
             .withColumn("loading_ts", _plus_hours(
                 _at(F.date_add("event_date", 1), "04:00:00"), F.col("leg") * F.lit(0.5)))
             .withColumn("depart_ts", _plus_hours(
                 _at(F.date_add("event_date", 1), "06:00:00"), F.col("leg") * F.lit(0.5))))
    return (timed
            .withColumn("eta", _plus_hours(F.col("depart_ts"), F.col("_travel_h")))
            .withColumn("unloading_ts", _plus_hours(F.col("eta"), F.lit(0.25)))
            .withColumn("etd", _plus_hours(F.col("eta"), F.col("_unload_h")))
            .withColumn("unload_minutes",
                        F.round(F.col("_unload_h") * F.lit(60.0), 1))
            .drop("_day_num", "slot", "n_trucks", "_travel_h", "_unload_h"))


def _truck_moves(shipments: DataFrame) -> DataFrame:
    stages = F.array(*[
        F.struct(F.lit(status).alias("status"), F.col(ts_col).alias("event_ts"))
        for status, ts_col in [
            ("SCHEDULED", "scheduled_ts"), ("LOADING", "loading_ts"),
            ("IN_TRANSIT", "depart_ts"), ("ARRIVED", "eta"),
            ("UNLOADING", "unloading_ts"), ("COMPLETED", "etd"),
        ]
    ])
    done = F.col("status") == "COMPLETED"
    return (shipments
            .withColumn("_s", F.explode(stages))
            .select(
                F.col("_s.event_ts").alias("event_ts"),
                F.col("truck_id"), F.col("dc_id"), F.col("store_id"),
                F.col("shipment_id"), F.col("_s.status").alias("status"),
                # eta/etd are known at scheduling -> populated on all rows;
                # departure_time/actual_unload_duration only once COMPLETED.
                F.col("eta"), F.col("etd"),
                F.col("unload_minutes"),
            )
            .withColumn("departure_time",
                        F.when(done, F.col("etd")).cast("timestamp"))
            .withColumn("actual_unload_duration",
                        F.when(done, F.col("unload_minutes")).cast("double"))
            .withColumn("trace_id", F.concat(
                F.lit("TRC-TM-"), F.col("shipment_id"), F.lit("-"),
                F.col("status")))
            .withColumn("event_date", F.to_date("event_ts")))


def _truck_inventory(shipments: DataFrame, reorders: DataFrame) -> DataFrame:
    products = reorders.select("store_id", "event_date", "leg", "product_id",
                               "reorder_quantity")
    actions = F.array(
        F.struct(F.lit("LOAD").alias("action"),
                 F.col("loading_ts").alias("event_ts"),
                 F.col("dc_id").alias("location_id"),
                 F.lit("DC").alias("location_type")),
        F.struct(F.lit("UNLOAD").alias("action"),
                 F.col("unloading_ts").alias("event_ts"),
                 F.col("store_id").alias("location_id"),
                 F.lit("STORE").alias("location_type")),
    )
    return (shipments.join(products, ["store_id", "event_date", "leg"])
            .withColumn("_a", F.explode(actions))
            .select(
                F.col("_a.event_ts").alias("event_ts"),
                "truck_id", "shipment_id", "product_id",
                F.col("reorder_quantity").alias("quantity"),
                F.col("_a.action").alias("action"),
                F.col("_a.location_id").alias("location_id"),
                F.col("_a.location_type").alias("location_type"),
                "dc_id", "store_id",
            )
            .withColumn("trace_id", F.concat(
                F.lit("TRC-TI-"), F.col("shipment_id"), F.lit("-"),
                F.col("product_id").cast("string"), F.lit("-"),
                F.col("action")))
            .withColumn("event_date", F.to_date("event_ts")))


# ---------------------------------------------------------------------------
# Stages 5-6: DC txns + store inbound mirrors
# ---------------------------------------------------------------------------

def _dc_txns(spark: SparkSession, truck_inv: DataFrame, n_products: int,
             d: seeded_draws, cfg: GenerationConfig) -> DataFrame:
    # --- supplier inbound: per (dc, day) explode 1-3 deliveries x 5 picks.
    days = (cfg.end_date - cfg.start_date).days + 1
    grid = spark.createDataFrame(
        [(dc, day) for dc in range(1, cfg.dc_count + 1) for day in range(days)],
        "dc_id long, _day_off long",
    ).withColumn("event_date",
                 F.date_add(F.lit(cfg.start_date), F.col("_day_off").cast("int")))
    n_ship = draw_int(d.u(["dc_id", "event_date"], "supplier-n"), 1, 3)
    keys = ["dc_id", "event_date", "seq", "pick"]
    inbound = (grid
               .withColumn("seq", F.explode(F.sequence(F.lit(1), n_ship)))
               .withColumn("pick", F.explode(F.sequence(F.lit(1), F.lit(5))))
               .withColumn("product_id",
                           (d.h64(keys, "supplier-prod") % F.lit(n_products)
                            + F.lit(1)).cast("long"))
               .withColumn("quantity", draw_int(d.u(keys, "supplier-qty"), 50, 500))
               .withColumn("source", F.concat(
                   F.lit("SUPPLIER-"), F.col("dc_id").cast("string"), F.lit("-"),
                   F.col("event_date").cast("string"), F.lit("-"),
                   F.col("seq").cast("string")))
               .withColumn("txn_type", F.lit("INBOUND_SHIPMENT"))
               # supplier delivery time-of-day: fixed 08:00 (documented choice)
               .withColumn("event_ts", _at(F.col("event_date"), "08:00:00"))
               # pick disambiguates duplicate uniform product picks per delivery
               .withColumn("trace_id", F.concat(
                   F.lit("TRC-DC-"), F.col("source"), F.lit("-"),
                   F.col("pick").cast("string"), F.lit("-"),
                   F.col("product_id").cast("string")))
               .withColumnRenamed("dc_id", "node_id")
               .select(*TXN_COLS))

    # --- outbound: mirror truck LOADs (negative qty, source = shipment_id).
    outbound = (truck_inv.filter(F.col("action") == "LOAD")
                .select(
                    F.col("dc_id").alias("node_id"), "product_id",
                    (-F.col("quantity")).cast("long").alias("quantity"),
                    F.lit("OUTBOUND_SHIPMENT").alias("txn_type"),
                    F.col("shipment_id").alias("source"),
                    "event_ts", "event_date",
                    F.concat(F.lit("TRC-DC-"), F.col("shipment_id"), F.lit("-"),
                             F.col("product_id").cast("string")).alias("trace_id"),
                ))
    return inbound.unionByName(outbound)


def _store_inbound(truck_inv: DataFrame) -> DataFrame:
    return (truck_inv.filter(F.col("action") == "UNLOAD")
            .select(
                F.col("store_id").alias("node_id"), "product_id",
                F.col("quantity").cast("long").alias("quantity"),
                F.lit("INBOUND_SHIPMENT").alias("txn_type"),
                F.col("shipment_id").alias("source"),
                "event_ts", "event_date",
                F.concat(F.lit("TRC-SI-"), F.col("shipment_id"), F.lit("-"),
                         F.col("product_id").cast("string")).alias("trace_id"),
            ))


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------

def generate_inventory_chain(
    spark: SparkSession,
    sales: dict[str, DataFrame],
    rets: dict[str, DataFrame],
    dims: dict[str, DataFrame],
    cfg: GenerationConfig,
) -> dict[str, DataFrame]:
    """Generate the six inventory/logistics fact tables (see module docstring)."""
    d = seeded_draws(cfg.seed)

    demand_txns = _sale_txns(sales, rets, d)
    reorders = _reorders(demand_txns, _store_dc_map(dims), d, cfg)
    shipments = _shipments(spark, reorders, dims, d, cfg)
    truck_moves = _truck_moves(shipments)
    truck_inv = _truck_inventory(shipments, reorders)

    n_products = dims["dim_products"].count()
    dc_raw = _dc_txns(spark, truck_inv, n_products, d, cfg)
    store_raw = demand_txns.unionByName(_store_inbound(truck_inv))

    store_bal = with_balances(store_raw, 40, 120, "ST", d, cfg)
    dc_bal = with_balances(dc_raw, 500, 2000, "DC", d, cfg)

    fact_store_txn = _with_index(
        store_bal.withColumnRenamed("node_id", "store_id"),
        "fact_store_inventory_txn")
    fact_dc_txn = _with_index(
        dc_bal.withColumnRenamed("node_id", "dc_id")
        # Rename lowercase source -> Source (TMDL-bound PascalCase). Keeping
        # both would be a case-insensitive duplicate that Delta rejects.
        .withColumnRenamed("source", "Source"),
        "fact_dc_inventory_txn")

    stockouts_df = (stockouts(store_bal, "ST", "StoreID")
                    .unionByName(stockouts(dc_bal, "DC", "DCID")))

    return {
        "fact_store_inventory_txn": fact_store_txn,
        "fact_dc_inventory_txn": fact_dc_txn,
        "fact_truck_moves": _with_index(truck_moves, "fact_truck_moves"),
        "fact_truck_inventory": _with_index(truck_inv, "fact_truck_inventory"),
        "fact_reorders": _with_index(reorders, "fact_reorders"),
        "fact_stockouts": _with_index(stockouts_df, "fact_stockouts"),
    }

# --- retail_setup/generation/gold.py ---
"""Gold aggregates — exact port of 02-historical-data-load.ipynb Part 3.

Operates on the in-memory ``GenerationResult.tables`` dict (``tables[...]``
replaces the legacy ``read_silver(...)``). Definitions are identical in
02-historical-data-load and 04-streaming-to-gold (verified 2026-06-12).

Deliberate decisions (vs the legacy notebooks):
- Money sums: legacy summed the formatted STRING columns (total_amount,
  subtotal_amount, tax_amount, ext_price, cost) relying on Spark's implicit
  cast — here we cast explicitly to double. Identical results, honest types.
- ``computed_at`` (top_products_15m) and ``as_of`` (both inventory positions)
  are produced exactly as the legacy code does, even though the TMDL doesn't
  bind them — extra columns are allowed.
- Legacy quirks preserved: truck_dwell_daily derives ``site`` as
  ``STORE_<id>`` / ``DC_<id>`` and filters ``dwell_min > 0``; the two
  inventory positions take the latest balance per key via a row_number
  window over ``event_ts`` descending.
"""

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


GOLD_TABLES: list[str] = [
    "sales_minute_store",
    "top_products_15m",
    "inventory_position_current",
    "dc_inventory_position_current",
    "truck_dwell_daily",
    "online_sales_daily",
    "zone_dwell_minute",
    "marketing_cost_daily",
    "tender_mix_daily",
]


def _money(col: str):
    """Explicit double cast for the legacy formatted string money columns."""
    return F.col(col).cast("double")


def generate_gold(
    spark: SparkSession, tables: dict[str, DataFrame]
) -> dict[str, DataFrame]:
    """Build the 9 Gold aggregate frames from the generated fact tables."""
    gold: dict[str, DataFrame] = {}

    # Sales by minute per store
    gold["sales_minute_store"] = (
        tables["fact_receipts"]
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "ts")
        .agg(
            F.sum(_money("total_amount")).alias("total_sales"),
            F.count("*").alias("receipts"),
            F.avg(_money("total_amount")).alias("avg_basket"),
        )
        .select(*column_names("sales_minute_store"))
    )

    # Top products by revenue (15m windows)
    gold["top_products_15m"] = (
        tables["fact_receipt_lines"]
        .withColumn("window_15m", F.window(F.col("event_ts"), "15 minutes"))
        .groupBy("product_id", "window_15m")
        .agg(
            F.sum(_money("ext_price")).alias("revenue"),
            F.sum("quantity").alias("units"),
        )
        .withColumn("computed_at", F.col("window_15m.end"))
        .drop("window_15m")
        .select(*column_names("top_products_15m"))
    )

    # Current store inventory position (latest balance per store/product)
    store_pos_window = Window.partitionBy("store_id", "product_id").orderBy(
        F.desc("event_ts"))
    gold["inventory_position_current"] = (
        tables["fact_store_inventory_txn"]
        .withColumn("rn", F.row_number().over(store_pos_window))
        .filter(F.col("rn") == 1)
        .select(
            "store_id", "product_id",
            F.col("balance").alias("on_hand"),
            F.col("event_ts").alias("as_of"),
        )
        .select(*column_names("inventory_position_current"))
    )

    # DC inventory position (latest balance per dc/product)
    dc_pos_window = Window.partitionBy("dc_id", "product_id").orderBy(
        F.desc("event_ts"))
    gold["dc_inventory_position_current"] = (
        tables["fact_dc_inventory_txn"]
        .withColumn("rn", F.row_number().over(dc_pos_window))
        .filter(F.col("rn") == 1)
        .select(
            "dc_id", "product_id",
            F.col("balance").alias("on_hand"),
            F.col("event_ts").alias("as_of"),
        )
        .select(*column_names("dc_inventory_position_current"))
    )

    # Truck dwell time daily
    gold["truck_dwell_daily"] = (
        tables["fact_truck_moves"]
        .withColumn("day", F.to_date("event_ts"))
        .withColumn(
            "site",
            F.when(F.col("store_id").isNotNull(),
                   F.concat(F.lit("STORE_"), F.col("store_id")))
            .otherwise(F.concat(F.lit("DC_"), F.col("dc_id"))),
        )
        .withColumn(
            "dwell_min",
            (F.unix_timestamp("etd") - F.unix_timestamp("eta")) / 60,
        )
        .filter(F.col("dwell_min").isNotNull() & (F.col("dwell_min") > 0))
        .groupBy("site", "day")
        .agg(
            F.avg("dwell_min").alias("avg_dwell_min"),
            F.countDistinct("truck_id").alias("trucks"),
        )
        .select(*column_names("truck_dwell_daily"))
    )

    # Online sales daily
    gold["online_sales_daily"] = (
        tables["fact_online_order_headers"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day")
        .agg(
            F.count("*").alias("orders"),
            F.sum(_money("subtotal_amount")).alias("subtotal"),
            F.sum(_money("tax_amount")).alias("tax"),
            F.sum(_money("total_amount")).alias("total"),
            F.avg(_money("total_amount")).alias("avg_order_value"),
        )
        .select(*column_names("online_sales_daily"))
    )

    # Zone dwell per minute
    gold["zone_dwell_minute"] = (
        tables["fact_foot_traffic"]
        .withColumn("ts", F.date_trunc("minute", F.col("event_ts")))
        .groupBy("store_id", "zone", "ts")
        .agg(
            F.avg("dwell_seconds").alias("avg_dwell"),
            F.sum("count").alias("customers"),
        )
        .select(*column_names("zone_dwell_minute"))
    )

    # Marketing cost daily
    gold["marketing_cost_daily"] = (
        tables["fact_marketing"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("campaign_id", "day")
        .agg(
            F.count("*").alias("impressions"),
            F.sum(_money("cost")).alias("cost"),
        )
        .select(*column_names("marketing_cost_daily"))
    )

    # Tender mix daily
    gold["tender_mix_daily"] = (
        tables["fact_receipts"]
        .withColumn("day", F.to_date("event_ts"))
        .groupBy("day", "payment_method")
        .agg(
            F.count("*").alias("transactions"),
            F.sum(_money("total_amount")).alias("total_amount"),
        )
        .select(*column_names("tender_mix_daily"))
    )

    return gold

# --- retail_setup/generation/invariants.py ---
"""Cross-table invariant checks. Pure reads; raises nothing — returns a report."""

from dataclasses import dataclass, field

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


@dataclass
class InvariantReport:
    checks: list[str] = field(default_factory=list)
    failures: list[str] = field(default_factory=list)
    row_counts: dict[str, int] = field(default_factory=dict)

    @property
    def passed(self) -> bool:
        return not self.failures


def _check(report: InvariantReport, name: str, bad_count: int) -> None:
    report.checks.append(name)
    if bad_count:
        report.failures.append(f"{name}: {bad_count} violations")


def run_invariants(spark: SparkSession, t: dict[str, DataFrame]) -> InvariantReport:
    r = InvariantReport()
    for name, df in t.items():
        r.row_counts[name] = df.count()

    receipts, lines, pay = t["fact_receipts"], t["fact_receipt_lines"], t["fact_payments"]
    _check(r, "fact_receipts.receipt_id_ext unique",
           r.row_counts["fact_receipts"] - receipts.select("receipt_id_ext").distinct().count())
    _check(r, "fact_receipt_lines -> fact_receipts FK",
           lines.join(receipts, "receipt_id_ext", "left_anti").count())
    _check(r, "fact_payments xor keys",
           pay.filter(F.col("receipt_id_ext").isNotNull() == F.col("order_id_ext").isNotNull()).count())
    _check(r, "fact_payments -> receipts FK",
           pay.filter(F.col("receipt_id_ext").isNotNull())
              .join(receipts, "receipt_id_ext", "left_anti").count())

    products = t["dim_products"].select(F.col("ID").alias("product_id"))
    for tbl in ["fact_receipt_lines", "fact_online_order_lines", "fact_promo_lines",
                "fact_store_inventory_txn", "fact_dc_inventory_txn", "fact_reorders"]:
        _check(r, f"{tbl} -> dim_products FK",
               t[tbl].join(products, "product_id", "left_anti").count())

    stores = t["dim_stores"].select(F.col("ID").alias("store_id"))
    for tbl in ["fact_receipts", "fact_store_inventory_txn", "fact_reorders",
                "fact_store_ops", "fact_foot_traffic", "fact_ble_pings"]:
        _check(r, f"{tbl} -> dim_stores FK",
               t[tbl].join(stores, "store_id", "left_anti").count())

    for tbl, df in t.items():
        if tbl.startswith("fact_") and "event_date" in df.columns:
            _check(r, f"{tbl}.event_date not null",
                   df.filter(F.col("event_date").isNull()).count())

    oh, ol = t["fact_online_order_headers"], t["fact_online_order_lines"]
    _check(r, "online order ids unique",
           r.row_counts["fact_online_order_headers"]
           - oh.select("order_id_ext").distinct().count())
    _check(r, "online lines -> headers FK",
           ol.join(oh, ol.order_id == oh.order_id_ext, "left_anti").count())

    so = t["fact_stockouts"]
    _check(r, "stockouts StoreID xor DCID",
           so.filter(F.col("StoreID").isNotNull() == F.col("DCID").isNotNull()).count())

    # Per-receipt promo discount consistency: for receipts present in
    # fact_promo_lines (SALE only by construction), the summed discount must
    # equal the implied line-level discount sum(unit_cents*quantity - ext_cents).
    promo_sum = (t["fact_promo_lines"]
                 .groupBy("receipt_id_ext")
                 .agg(F.sum("discount_cents").alias("promo_discount")))
    line_sum = (lines
                .groupBy("receipt_id_ext")
                .agg(F.sum(F.col("unit_cents") * F.col("quantity")
                           - F.col("ext_cents")).alias("line_discount")))
    _check(r, "promo discount consistency",
           promo_sum.join(line_sum, "receipt_id_ext", "left")
           .filter(F.col("line_discount").isNull()
                   | (F.col("promo_discount") != F.col("line_discount")))
           .count())

    # --- dimension geography FK integrity (datagen foreign_key validator parity)
    geo_ids = t["dim_geographies"].select(F.col("ID").alias("geo_id"))
    for dim in ["dim_stores", "dim_distribution_centers", "dim_customers"]:
        _check(r, f"{dim} -> dim_geographies FK",
               t[dim].select(F.col("GeographyID").alias("geo_id"))
               .join(geo_ids, "geo_id", "left_anti").count())

    # --- DC coverage on facts that reference a distribution center
    dc_ids = t["dim_distribution_centers"].select(F.col("ID").alias("dc_id"))
    for tbl in ["fact_dc_inventory_txn", "fact_truck_moves", "fact_reorders"]:
        _check(r, f"{tbl} -> dim_distribution_centers FK",
               t[tbl].filter(F.col("dc_id").isNotNull()).select("dc_id")
               .join(dc_ids, "dc_id", "left_anti").count())

    # --- truck coverage on logistics facts
    truck_ids = t["dim_trucks"].select(F.col("ID").alias("truck_id"))
    for tbl in ["fact_truck_moves", "fact_truck_inventory"]:
        _check(r, f"{tbl} -> dim_trucks FK",
               t[tbl].filter(F.col("truck_id").isNotNull()).select("truck_id")
               .join(truck_ids, "truck_id", "left_anti").count())

    # --- truck timing: arrival (eta) must not be after completion (etd)
    tm = t["fact_truck_moves"]
    _check(r, "fact_truck_moves etd >= eta",
           tm.filter(F.col("eta").isNotNull() & F.col("etd").isNotNull()
                     & (F.col("etd") < F.col("eta"))).count())

    # --- customer coverage on facts that resolve a customer (nullable for some)
    customer_ids = t["dim_customers"].select(F.col("ID").alias("customer_id"))
    for tbl in ["fact_receipts", "fact_online_order_headers"]:
        _check(r, f"{tbl} -> dim_customers FK",
               t[tbl].filter(F.col("customer_id").isNotNull()).select("customer_id")
               .join(customer_ids, "customer_id", "left_anti").count())
    # fact_marketing.customer_id is a nullable double (low resolution rate); cast.
    _check(r, "fact_marketing -> dim_customers FK",
           t["fact_marketing"].filter(F.col("customer_id").isNotNull())
           .select(F.col("customer_id").cast("long").alias("customer_id"))
           .join(customer_ids, "customer_id", "left_anti").count())

    # --- dim_products pricing constraints (datagen pricing validator parity)
    prod = t["dim_products"]
    _check(r, "dim_products pricing Cost<SalePrice<=MSRP",
           prod.filter(~((F.col("Cost") > 0)
                         & (F.col("Cost") < F.col("SalePrice"))
                         & (F.col("SalePrice") <= F.col("MSRP")))).count())
    return r

# --- retail_setup/generation/engine.py ---
"""Orchestrates full generation. Returns DataFrames; writing happens in 2c."""

from dataclasses import dataclass
from datetime import date

from pyspark.sql import DataFrame, SparkSession



@dataclass
class GenerationResult:
    tables: dict[str, DataFrame]


def _shift_year(d: date, years: int) -> date:
    """Shift a date by whole years; Feb 29 falls back to Feb 28."""
    try:
        return d.replace(year=d.year + years)
    except ValueError:
        return date(d.year + years, d.month, 28)


def generate_all(
    spark: SparkSession, dicts: DictionarySet, cfg: GenerationConfig
) -> GenerationResult:
    t: dict[str, DataFrame] = {}
    t.update(generate_dimensions(spark, dicts, cfg))
    t["dim_date"] = generate_dim_date(
        spark, _shift_year(cfg.start_date, -5), _shift_year(cfg.end_date, 5))

    sales = generate_receipts_group(spark, t, dicts.profile, cfg)
    # fact_receipts/lines (SALE-only) each feed several independent builders —
    # returns, promotions, foot traffic, BLE, inventory — plus the SALE/RETURN
    # unions below. Persist them so this shared, expensive lineage (xxhash draws
    # + line explode) is computed once instead of once per consumer. Generation
    # is fully deterministic, so a cached frame is byte-identical to a recomputed
    # one: realism is unchanged, only the redundant recomputation is removed.
    sales["fact_receipts"] = sales["fact_receipts"].cache()
    sales["fact_receipt_lines"] = sales["fact_receipt_lines"].cache()
    rets = generate_returns(spark, sales, t, cfg)
    t["fact_receipts"] = sales["fact_receipts"].unionByName(rets["fact_receipts"])
    t["fact_receipt_lines"] = sales["fact_receipt_lines"].unionByName(
        rets["fact_receipt_lines"])

    online = generate_online_orders(spark, t, dicts.profile, cfg)
    t["fact_online_order_headers"] = online["fact_online_order_headers"]
    t["fact_online_order_lines"] = online["fact_online_order_lines"]
    # single-writer union for the shared payments table (2a carry-note)
    t["fact_payments"] = (sales["fact_payments"]
                          .unionByName(rets["fact_payments"])
                          .unionByName(online["payments"]))

    promos, promo_lines = generate_promotions(spark, sales)
    t["fact_promotions"], t["fact_promo_lines"] = promos, promo_lines
    t["fact_marketing"] = generate_marketing(spark, t, cfg)
    t["fact_store_ops"] = generate_store_ops(spark, t, cfg)
    t["fact_foot_traffic"] = generate_foot_traffic(
        spark, sales["fact_receipts"], t, cfg)
    pings, zc = generate_ble(spark, sales["fact_receipts"], t, cfg)
    t["fact_ble_pings"], t["fact_customer_zone_changes"] = pings, zc
    t.update(generate_inventory_chain(spark, sales, rets, t, cfg))
    # Downstream the driver runs run_invariants (50+ count/join/distinct actions
    # over these frames) and then write_all (one write + count per table).
    # Without caching, every one of those actions re-executes the full generation
    # DAG from scratch — the dominant cost of the setup run. Persist each table so
    # it materializes exactly once (on the first invariant pass) and all later
    # reads hit the cache. Deterministic generation ⇒ cached == recomputed, so
    # the simulation output is identical. cache() is MEMORY_AND_DISK, so large
    # frames spill to local SSD rather than failing under memory pressure.
    for name in t:
        t[name] = t[name].cache()
    return GenerationResult(tables=t)

# --- retail_setup/generation/writer.py ---
"""Thin write layer. Notebooks call write_to_lakehouse; tests use write_table
with a format/location override (no delta-spark dependency locally)."""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F



def write_table(df: DataFrame, location: str, fmt: str = "delta") -> None:
    df.write.format(fmt).mode("overwrite").save(location)


def write_to_lakehouse(df: DataFrame, lakehouse: str, schema: str, table: str) -> None:
    """Overwrite-by-design, matching 02-historical-data-load semantics."""
    df.write.format("delta").mode("overwrite").saveAsTable(f"{lakehouse}.{schema}.{table}")


def write_all(
    tables: dict[str, DataFrame],
    gold: dict[str, DataFrame],
    cfg: GenerationConfig,
    run_id: str,
    *,
    lakehouse: str | None = None,
    base_path: str | None = None,
    fmt: str = "delta",
) -> list[str]:
    """Persist dims+facts to silver, gold to gold, then setup_run_log.

    Two modes: catalog (lakehouse=...) used by notebooks —
    saveAsTable(f"{lakehouse}.{cfg.silver_db}.{name}") — or path
    (base_path=...) used by local tests/E2E — save(f"{base_path}/{db}/{name}").
    dim_date is written as generated by the engine (±5y padding preserved —
    the semantic model's date relationships need the headroom).
    Returns the list of written table names (silver + gold); the
    setup_run_log table itself is not included in the returned list.

    The Spark session is derived from the first DataFrame in ``tables``
    (``df.sparkSession``) — no explicit session parameter is needed.
    """
    if (lakehouse is None) == (base_path is None):
        raise ValueError("Provide exactly one of lakehouse= or base_path=")

    spark = next(iter(tables.values())).sparkSession

    if lakehouse is not None:
        for db in (cfg.silver_db, cfg.gold_db):
            spark.sql(f"CREATE DATABASE IF NOT EXISTS {lakehouse}.{db}")

    def _write(df: DataFrame, db: str, name: str) -> None:
        if lakehouse is not None:
            df.write.format("delta").mode("overwrite").saveAsTable(
                f"{lakehouse}.{db}.{name}"
            )
        else:
            df.write.format(fmt).mode("overwrite").save(f"{base_path}/{db}/{name}")

    def _count(db: str, name: str) -> int:
        # Count the just-written table, not the in-memory df: df.count() would
        # re-execute the entire (deterministic) generation lineage a second time,
        # while a freshly written Delta/Parquet table answers count() from file
        # metadata. Same value, a fraction of the work.
        if lakehouse is not None:
            return spark.table(f"{lakehouse}.{db}.{name}").count()
        return spark.read.format(fmt).load(f"{base_path}/{db}/{name}").count()

    written: list[tuple[str, int]] = []
    for name, df in tables.items():
        _write(df, cfg.silver_db, name)
        written.append((name, _count(cfg.silver_db, name)))
    for name, df in gold.items():
        _write(df, cfg.gold_db, name)
        written.append((name, _count(cfg.gold_db, name)))

    log_rows = [
        (run_id, cfg.store_type, cfg.seed, cfg.start_date, cfg.end_date,
         name, count)
        for name, count in written
    ]
    log_df = spark.createDataFrame(
        log_rows,
        "run_id string, store_type string, seed long, start_date date, "
        "end_date date, table_name string, row_count long",
    ).withColumn("generated_at", F.current_timestamp())
    _write(log_df, cfg.silver_db, "setup_run_log")

    return [name for name, _ in written]

## Generate and write dimensions

In [ ]:
from datetime import date

cfg = GenerationConfig(
    store_type=STORE_TYPE,
    start_date=date.fromisoformat(START_DATE),
    end_date=date.fromisoformat(END_DATE),
    store_count=STORE_COUNT,
    seed=SEED,
    silver_db=SILVER_DB,
    gold_db=GOLD_DB,
    dictionary_root="/lakehouse/default/Files/setup/dictionaries",
)
dicts = load_dictionaries(cfg.resolved_dictionary_root, cfg.store_type)

dims = generate_dimensions(spark, dicts, cfg)
# dim_date is padded ±5 years — the semantic model's date relationships
# need the headroom around the generated fact window.
dims["dim_date"] = generate_dim_date(
    spark, _shift_year(cfg.start_date, -5), _shift_year(cfg.end_date, 5))

spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{SILVER_DB}")
for name, df in dims.items():
    write_to_lakehouse(df, LAKEHOUSE_NAME, SILVER_DB, name)
    # Count the written table, not df: df.count() would recompute the dimension
    # generation; the persisted Delta table answers count() from metadata.
    n = spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{name}").count()
    print(f"wrote {LAKEHOUSE_NAME}.{SILVER_DB}.{name}: {n:,} rows")